<a href="https://colab.research.google.com/github/franzmir-oss/tradingmonitor/blob/main/Copia_de_Untitled0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print("Mi agente de trading está funcionando 🚀")


Mi agente de trading está funcionando 🚀


In [2]:
!pip install yfinance -q


In [3]:
import yfinance as yf

spy = yf.download("SPY", period="5d", interval="5m")

spy.tail()

/tmp/ipykernel_3676/4080369549.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  spy = yf.download("SPY", period="5d", interval="5m")
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,SPY,SPY,SPY,SPY,SPY
Datetime,,,,,
2026-08-28 19:35:00+00:00,769.544983,769.575012,769.130005,769.400024,390876
2026-08-28 19:40:00+00:00,769.619995,769.789978,769.530029,769.549988,701568
2026-08-28 19:45:00+00:00,769.239990,769.690002,769.020020,769.599976,802394
2026-08-28 19:50:00+00:00,769.359985,769.780029,769.109985,769.270020,1057313
2026-08-28 19:55:00+00:00,769.380005,769.880005,769.070007,769.349976,2670344


In [4]:
import yfinance as yf
import pandas as pd

# Descargar datos de SPY
spy = yf.download(
    "SPY",
    period="5d",
    interval="5m",
    auto_adjust=False
)

# Quitamos niveles innecesarios de columnas si aparecen
if isinstance(spy.columns, pd.MultiIndex):
    spy.columns = spy.columns.get_level_values(0)

# EMA 9 y EMA 21
spy["EMA_9"] = spy["Close"].ewm(span=9, adjust=False).mean()
spy["EMA_21"] = spy["Close"].ewm(span=21, adjust=False).mean()

# RSI 14
delta = spy["Close"].diff()

ganancias = delta.clip(lower=0)
perdidas = -delta.clip(upper=0)

media_ganancias = ganancias.rolling(14).mean()
media_perdidas = perdidas.rolling(14).mean()

RS = media_ganancias / media_perdidas
spy["RSI_14"] = 100 - (100 / (1 + RS))

# Mostrar los últimos datos
spy[["Close", "EMA_9", "EMA_21", "RSI_14", "Volume"]].tail(10)


[*********************100%***********************]  1 of 1 completed


Price,Close,EMA_9,EMA_21,RSI_14,Volume
Datetime,,,,,
2026-08-28 19:10:00+00:00,769.236328,769.063615,769.246633,45.807930,373847
2026-08-28 19:15:00+00:00,769.210022,769.092897,769.243305,44.643052,298514
2026-08-28 19:20:00+00:00,769.320007,769.138319,769.250278,51.603185,495086
2026-08-28 19:25:00+00:00,769.515015,769.213658,769.274345,51.603185,260564
2026-08-28 19:30:00+00:00,769.400024,769.250931,769.285770,55.709730,395715
2026-08-28 19:35:00+00:00,769.544983,769.309742,769.309335,58.016546,390876
2026-08-28 19:40:00+00:00,769.619995,769.371792,769.337577,54.480420,701568
2026-08-28 19:45:00+00:00,769.239990,769.345432,769.328705,52.233828,802394
2026-08-28 19:50:00+00:00,769.359985,769.348343,769.331549,50.796189,1057313


In [5]:
import pandas as pd
import yfinance as yf

# 1. Descarga de datos horarios (60m) para SPY (Enfoque institucional)
print("Descargando datos horarios de SPY...")
spy = yf.download("SPY", period="60d", interval="60m", auto_adjust=False)

# Limpieza de columnas si yfinance devuelve multiíndice
if isinstance(spy.columns, pd.MultiIndex):
  spy.columns = spy.columns.get_level_values(0)

# 2. Cálculo de la SMA 40 (Clave para los pullbacks del Módulo 4 y 5)
spy["SMA_40"] = spy["Close"].rolling(window=40).mean()

# 3. Extraer la hora de la vela para identificar la sesión y la Regla de las 11:00 AM
spy["Hora"] = spy.index.hour

# 4. Mostrar la estructura de los últimos datos con la SMA 40
print("\n--- Últimos registros en Marco Horario (60m) ---")
print(spy[["Close", "SMA_40", "Volume", "Hora"]].tail(12))


Descargando datos horarios de SPY...


[*********************100%***********************]  1 of 1 completed


--- Últimos registros en Marco Horario (60m) ---
Price                           Close      SMA_40    Volume  Hora
Datetime                                                         
2026-08-27 15:30:00+00:00  770.909973  765.659380   2969743    15
2026-08-27 16:30:00+00:00  771.750000  765.712379   2858782    16
2026-08-27 17:30:00+00:00  769.460022  765.721629   3207478    17
2026-08-27 18:30:00+00:00  769.940002  765.814630   4031698    18
2026-08-27 19:30:00+00:00  771.039978  765.921880   5493749    19
2026-08-28 13:30:00+00:00  771.299988  766.060754   8479687    13
2026-08-28 14:30:00+00:00  774.950012  766.315004  25412763    14
2026-08-28 15:30:00+00:00  769.599976  766.466003  20395572    15
2026-08-28 16:30:00+00:00  768.820007  766.599254   3734084    16
2026-08-28 17:30:00+00:00  769.059998  766.759753  23768857    17
2026-08-28 18:30:00+00:00  769.515015  766.881378  26916873    18
2026-08-28 19:30:00+00:00  769.380005  766.967003   6018210    19


In [6]:
# 1. Convertir el índice de las velas a la hora oficial de Nueva York
spy.index = spy.index.tz_convert('America/New_York')

# 2. Actualizar la columna de hora con el horario de NY
spy["Hora_NY"] = spy.index.hour

# 3. Aislar estrictamente la vela de las 11:00 AM (Regla de las 11:00 AM)
vela_11am = spy[spy["Hora_NY"] == 11].copy()

# 4. Clasificar si la vela de las 11:00 AM es verde (alcista) o roja (bajista)
vela_11am["Efecto_11AM"] = vela_11am.apply(
    lambda row: 'VERDE 🟢 (Predisposición Alcista)' if row['Close'] > row['Open'] else 'ROJA 🔴 (Predisposición Bajista)',
    axis=1
)

# 5. Mostrar el histórico reciente de la regla de las 11:00 AM
print("--- EVALUACIÓN DE LA REGLA DE LAS 11:00 AM ---")
print(vela_11am[['Open', 'Close', 'Efecto_11AM', 'Volume']].tail(8))

--- EVALUACIÓN DE LA REGLA DE LAS 11:00 AM ---
Price                            Open       Close  \
Datetime                                            
2026-08-19 11:30:00-04:00  771.016296  770.669983   
2026-08-20 11:30:00-04:00  766.739990  765.744995   
2026-08-21 11:30:00-04:00  765.960022  766.515015   
2026-08-24 11:30:00-04:00  763.739990  764.890015   
2026-08-25 11:30:00-04:00  765.320007  765.760010   
2026-08-26 11:30:00-04:00  765.840027  764.799988   
2026-08-27 11:30:00-04:00  770.309998  770.909973   
2026-08-28 11:30:00-04:00  774.969971  769.599976   

Price                                           Efecto_11AM    Volume  
Datetime                                                               
2026-08-19 11:30:00-04:00   ROJA 🔴 (Predisposición Bajista)   3117626  
2026-08-20 11:30:00-04:00   ROJA 🔴 (Predisposición Bajista)   3494078  
2026-08-21 11:30:00-04:00  VERDE 🟢 (Predisposición Alcista)   7924032  
2026-08-24 11:30:00-04:00  VERDE 🟢 (Predisposición Alcista)   

In [7]:
# 1. Calcular el bloque completo de Medias Móviles Simples (SMA)
spy["SMA_20"] = spy["Close"].rolling(window=20).mean()
spy["SMA_40"] = spy["Close"].rolling(window=40).mean()
spy["SMA_100"] = spy["Close"].rolling(window=100).mean()
spy["SMA_200"] = spy["Close"].rolling(window=200).mean()

# 2. Definir Alineación en "Líneas de Ferrocarril" Alcista (Módulo 4)
# SMA 20 > SMA 40 > SMA 100 > SMA 200
spy["Tendencia_Alcista"] = (
    (spy["SMA_20"] > spy["SMA_40"]) &
    (spy["SMA_40"] > spy["SMA_100"]) &
    (spy["SMA_100"] > spy["SMA_200"])
)

# 3. Detectar Pullback: cuando el precio toca o se acerca milimétricamente a la SMA 40 (margen del 0.4%)
spy["Distancia_SMA40"] = abs(spy["Low"] - spy["SMA_40"]) / spy["SMA_40"]
spy["Pullback_Alcista_Detectado"] = spy["Tendencia_Alcista"] & (spy["Distancia_SMA40"] <= 0.004)

# 4. Filtrar y mostrar las veces que el sistema ha detectado esta configuración exacta
resultados_pullback = spy[spy["Pullback_Alcista_Detectado"]].copy()

print("--- DETECCIONES DE PULLBACK A LA SMA 40 (TENDENCIA ALCISTA) ---")
if not resultados_pullback.empty:
    print(resultados_pullback[["Close", "SMA_20", "SMA_40", "Volume"]].tail(5))
    print(f"\nTotal de pullbacks alcistas detectados en el histórico cargado: {len(resultados_pullback)}")
else:
    print("No hay coincidencias exactas en este rango de fechas. Vamos a revisar la sensibilidad del rango.")

--- DETECCIONES DE PULLBACK A LA SMA 40 (TENDENCIA ALCISTA) ---
Price                           Close      SMA_20      SMA_40   Volume
Datetime                                                              
2026-08-17 11:30:00-04:00  775.289978  776.070258  774.266240  2480472
2026-08-17 12:30:00-04:00  773.580017  776.094757  774.278865  2932105
2026-08-17 13:30:00-04:00  773.590027  776.125259  774.290616  3486505
2026-08-17 14:30:00-04:00  773.119995  776.155258  774.259741  3522124
2026-08-17 15:30:00-04:00  772.679993  775.829758  774.233241  8585280

Total de pullbacks alcistas detectados en el histórico cargado: 31


In [8]:
# 5. GATILLO DE ENTRADA (Confirmación de la Estrategia)
# Identificamos el pullback registrado en la vela anterior
spy["Pullback_Anterior"] = spy["Pullback_Alcista_Detectado"].shift(1)

# Condición de confirmación:
# A) La vela actual es verde sólida (Close > Open)
spy["Vela_Verde"] = spy["Close"] > spy["Open"]

# B) El precio rompe al alza el máximo de la vela de pullback anterior
spy["Ruptura_Maximo"] = spy["High"] > spy["High"].shift(1)

# 6. SEÑAL DEFINITIVA DE COMPRA (CALL)
spy["Señal_CALL"] = spy["Pullback_Anterior"] & spy["Vela_Verde"] & spy["Ruptura_Maximo"]

# Filtrar y mostrar las señales exactas de disparo
entradas_call = spy[spy["Señal_CALL"]].copy()

print("--- SEÑALES EXACTAS DE ENTRADA (CALL) ---")
if not entradas_call.empty:
    print(entradas_call[["Close", "SMA_40", "Volume"]].tail(5))
    print(f"\nTotal de señales de entrada CALL detectadas: {len(entradas_call)}")
else:
    print("No hay señales con la confirmación estricta en este tramo de fechas.")

--- SEÑALES EXACTAS DE ENTRADA (CALL) ---
Price                           Close      SMA_40   Volume
Datetime                                                  
2026-08-12 12:30:00-04:00  772.780029  771.711758  2563103
2026-08-12 13:30:00-04:00  773.090027  771.757008  2417715
2026-08-13 09:30:00-04:00  779.190002  771.909486  6564966
2026-08-14 14:30:00-04:00  776.155029  773.920740  3082814

Total de señales de entrada CALL detectadas: 4


In [9]:
# MÓDULO 5: Pullback a la SMA 40 en Tendencia Bajista (PUTs)

# 1. Definir Alineación en "Líneas de Ferrocarril" Bajista
# SMA 20 < SMA 40 < SMA 100 < SMA 200
spy["Tendencia_Bajista"] = (
    (spy["SMA_20"] < spy["SMA_40"]) &
    (spy["SMA_40"] < spy["SMA_100"]) &
    (spy["SMA_100"] < spy["SMA_200"])
)

# 2. Detectar Pullback: cuando el precio rebota y se acerca milimétricamente a la SMA 40 desde abajo
spy["Distancia_SMA40_Bajista"] = abs(spy["High"] - spy["SMA_40"]) / spy["SMA_40"]
spy["Pullback_Bajista_Detectado"] = spy["Tendencia_Bajista"] & (spy["Distancia_SMA40_Bajista"] <= 0.004)

# 3. GATILLO DE ENTRADA (Confirmación Bajista)
# Identificamos el pullback registrado en la vela anterior
spy["Pullback_Bajista_Anterior"] = spy["Pullback_Bajista_Detectado"].shift(1)

# Condición de confirmación para PUT:
# A) La vela actual es roja sólida (Close < Open)
spy["Vela_Roja"] = spy["Close"] < spy["Open"]

# B) El precio rompe a la baja el mínimo de la vela de pullback anterior
spy["Ruptura_Minimo"] = spy["Low"] < spy["Low"].shift(1)

# 4. SEÑAL DEFINITIVA DE COMPRA (PUT)
spy["Señal_PUT"] = spy["Pullback_Bajista_Anterior"] & spy["Vela_Roja"] & spy["Ruptura_Minimo"]

# Filtrar y mostrar las señales exactas de disparo
entradas_put = spy[spy["Señal_PUT"]].copy()

print("--- SEÑALES EXACTAS DE ENTRADA (PUT) ---")
if not entradas_put.empty:
    print(entradas_put[["Close", "SMA_40", "Volume"]].tail(5))
    print(f"\nTotal de señales de entrada PUT detectadas: {len(entradas_put)}")
else:
    print("No hay señales de PUT con la confirmación estricta en este tramo de fechas.")

--- SEÑALES EXACTAS DE ENTRADA (PUT) ---
No hay señales de PUT con la confirmación estricta en este tramo de fechas.


In [10]:
import pandas as pd
import yfinance as yf

# --- SCRIPT MAESTRO DE EVALUACIÓN Y GESTIÓN DE RIESGO ---

# 1. Función de Cálculo Matemático de Contratos y Take Profit (Módulo 6)
# Basado en la fórmula exacta del libro: Duplicar capital neto de comisiones ($7 + $1.25 por contrato)
def calcular_take_profit(num_contratos=10, precio_contrato_inicial=0.30):
    comision_total = 7.00 + (1.25 * num_contratos)
    inversion_neta = (num_contratos * precio_contrato_inicial) + comision_total
    costo_unitario_real = inversion_neta / num_contratos
    take_profit_limit = costo_unitario_real * 2  # Duplicar el capital invertido
    return costo_unitario_real, take_profit_limit

# 2. Resumen de Señales CALL detectadas y su Plan de Gestión
print("==================================================")
print("     INFORME AUTOMATIZADO - PLAN MAESTRO SPY      ")
print("==================================================")

if 'entradas_call' in locals() and not entradas_call.empty:
    print(f"\n🟢 Se han validado {len(entradas_call)} señales de compra (CALL):\n")
    for fecha, fila in entradas_call.iterrows():
        costo_real, tp = calcular_take_profit()
        print(f"📅 Fecha/Hora: {fecha}")
        print(f"   • Precio SPY: {fila['Close']:.2f} | SMA 40: {fila['SMA_40']:.2f}")
        print(f"   • Volumen institucional: {fila['Volume']:,}")
        print(f"   • 🎯 Configuración Take Profit (Limit Price): ${tp:.3f} por contrato\n")
else:
    print("\n🟢 No hay señales CALL pendientes de ejecución en el filtro actual.")

if 'entradas_put' in locals() and not entradas_put.empty:
    print(f"\n🔴 Se han validado {len(entradas_put)} señales de venta (PUT):\n")
    for fecha, fila in entradas_put.iterrows():
        costo_real, tp = calcular_take_profit()
        print(f"📅 Fecha/Hora: {fecha}")
        print(f"   • Precio SPY: {fila['Close']:.2f} | SMA 40: {fila['SMA_40']:.2f}")
        print(f"   • 🎯 Configuración Take Profit (Limit Price): ${tp:.3f} por contrato\n")
else:
    print("🔴 Sin señales PUT activas en este rango (Mercado en tendencia alcista principal).")

print("==================================================")
print("Sistema sincronizado con la Regla de las 11:00 AM y Módulos 4-6 🚀")

     INFORME AUTOMATIZADO - PLAN MAESTRO SPY      

🟢 Se han validado 4 señales de compra (CALL):

📅 Fecha/Hora: 2026-08-12 12:30:00-04:00
   • Precio SPY: 772.78 | SMA 40: 771.71
   • Volumen institucional: 2,563,103
   • 🎯 Configuración Take Profit (Limit Price): $4.500 por contrato

📅 Fecha/Hora: 2026-08-12 13:30:00-04:00
   • Precio SPY: 773.09 | SMA 40: 771.76
   • Volumen institucional: 2,417,715
   • 🎯 Configuración Take Profit (Limit Price): $4.500 por contrato

📅 Fecha/Hora: 2026-08-13 09:30:00-04:00
   • Precio SPY: 779.19 | SMA 40: 771.91
   • Volumen institucional: 6,564,966
   • 🎯 Configuración Take Profit (Limit Price): $4.500 por contrato

📅 Fecha/Hora: 2026-08-14 14:30:00-04:00
   • Precio SPY: 776.16 | SMA 40: 773.92
   • Volumen institucional: 3,082,814
   • 🎯 Configuración Take Profit (Limit Price): $4.500 por contrato

🔴 Sin señales PUT activas en este rango (Mercado en tendencia alcista principal).
Sistema sincronizado con la Regla de las 11:00 AM y Módulos 4-6 🚀


In [11]:
import pandas as pd
import yfinance as yf

# Universo completo de activos, ETFs y empresas de referencia del Plan Maestro
universo_activos = [
    "SPY",
    "QQQ",
    "DIA",  # ETFs Principales e Índices
    "AAPL",
    "AMZN",
    "GOOG",
    "MSFT",
    "NFLX",  # Tecnología y Crecimiento
    "CMG",
    "WMT",
    "COST",
    "MCD",
    "NKE",  # Consumo y Retail
    "JPM",
    "BAC",
    "WFC",
    "C",  # Banca y Finanzas
    "JNJ",
    "KO",
    "PEP",
    "F",  # Salud, Valor e Industriales
]

print(
    "Iniciando escaneo institucional multiactivo para todo el universo del Plan"
    " Maestro...\n"
)

resultados_calls = []
resultados_puts = []


def calcular_take_profit(num_contratos=10, precio_contrato_inicial=0.30):
  comision_total = 7.00 + (1.25 * num_contratos)
  inversion_neta = (num_contratos * precio_contrato_inicial) + comision_total
  costo_unitario_real = inversion_neta / num_contratos
  return costo_unitario_real * 2  # Duplicar capital neto


for ticker in universo_activos:
  try:
    # Descargar datos horarios en bloque institucional
    df = yf.download(ticker, period="60d", interval="60m", progress=False)
    if df.empty:
      continue

    # Limpieza de columnas si yfinance devuelve multiíndice
    if isinstance(df.columns, pd.MultiIndex):
      df.columns = df.columns.get_level_values(0)

    # Sincronización horaria a Nueva York
    df.index = df.index.tz_convert("America/New_York")
    df["Hora_NY"] = df.index.hour

    # Bloque de Medias Móviles Simples (SMA)
    df["SMA_20"] = df["Close"].rolling(window=20).mean()
    df["SMA_40"] = df["Close"].rolling(window=40).mean()
    df["SMA_100"] = df["Close"].rolling(window=100).mean()
    df["SMA_200"] = df["Close"].rolling(window=200).mean()

    # --- ESTRATEGIA ALCISTA: Pullback SMA 40 + Ferrocarril (CALL) ---
    tendencia_alcista = (
        (df["SMA_20"] > df["SMA_40"])
        & (df["SMA_40"] > df["SMA_100"])
        & (df["SMA_100"] > df["SMA_200"])
    )
    distancia_alcista = abs(df["Low"] - df["SMA_40"]) / df["SMA_40"]
    pullback_alcista = tendencia_alcista & (distancia_alcista <= 0.004)

    df["Pull_Anterior"] = pullback_alcista.shift(1)
    df["Vela_Verde"] = df["Close"] > df["Open"]
    df["Ruptura_Max"] = df["High"] > df["High"].shift(1)
    df["Señal_CALL"] = (
        df["Pull_Anterior"] & df["Vela_Verde"] & df["Ruptura_Max"]
    )

    # --- ESTRATEGIA BAJISTA: Pullback SMA 40 + Ferrocarril Invertido (PUT) ---
    tendencia_bajista = (
        (df["SMA_20"] < df["SMA_40"])
        & (df["SMA_40"] < df["SMA_100"])
        & (df["SMA_100"] < df["SMA_200"])
    )
    distancia_bajista = abs(df["High"] - df["SMA_40"]) / df["SMA_40"]
    pullback_bajista = tendencia_bajista & (distancia_bajista <= 0.004)

    df["Pull_Baj_Anterior"] = pullback_bajista.shift(1)
    df["Vela_Roja"] = df["Close"] < df["Open"]
    df["Ruptura_Min"] = df["Low"] < df["Low"].shift(1)
    df["Señal_PUT"] = (
        df["Pull_Baj_Anterior"] & df["Vela_Roja"] & df["Ruptura_Min"]
    )

    # Recolectar señales encontradas
    calls_act = df[df["Señal_CALL"]]
    puts_act = df[df["Señal_PUT"]]

    for idx, row in calls_act.iterrows():
      resultados_calls.append({
          "Ticker": ticker,
          "Fecha": idx,
          "Precio": row["Close"],
          "SMA_40": row["SMA_40"],
          "Volumen": row["Volume"],
          "TP": calcular_take_profit(),
      })

    for idx, row in puts_act.iterrows():
      resultados_puts.append({
          "Ticker": ticker,
          "Fecha": idx,
          "Precio": row["Close"],
          "SMA_40": row["SMA_40"],
          "Volumen": row["Volume"],
          "TP": calcular_take_profit(),
      })

  except Exception as e:
    continue

# --- INFORME GLOBAL DE RESULTADOS ---
print(
    "=========================================================================="
)
print("             ESCANEO GLOBAL MULTIACTIVO - PLAN MAESTRO                    ")
print(
    "=========================================================================="
)

if resultados_calls:
  print(f"\n🟢 TOTAL DE SEÑALES CALL DETECTADAS: {len(resultados_calls)}\n")
  for item in resultados_calls:
    print(
        f"[{item['Ticker']}] 📅 {item['Fecha']} | Precio: {item['Precio']:.2f}"
        f" | SMA 40: {item['SMA_40']:.2f} | Vol: {item['Volumen']:,} | 🎯 TP"
        f" Limit: ${item['TP']:.3f}"
    )
else:
  print("\n🟢 No hay señales CALL activas en este momento para el universo.")

if resultados_puts:
  print(f"\n🔴 TOTAL DE SEÑALES PUT DETECTADAS: {len(resultados_puts)}\n")
  for item in resultados_puts:
    print(
        f"[{item['Ticker']}] 📅 {item['Fecha']} | Precio: {item['Precio']:.2f}"
        f" | SMA 40: {item['SMA_40']:.2f} | Vol: {item['Volumen']:,} | 🎯 TP"
        f" Limit: ${item['TP']:.3f}"
    )
else:
  print("\n🔴 No hay señales PUT activas en este momento para el universo.")

print(
    "\n=========================================================================="
)
print("Análisis completado para todos los activos y ETFs del PDF 🚀")

Iniciando escaneo institucional multiactivo para todo el universo del Plan Maestro...



/tmp/ipykernel_3676/1137262478.py:48: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="60d", interval="60m", progress=False)
/tmp/ipykernel_3676/1137262478.py:48: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="60d", interval="60m", progress=False)
/tmp/ipykernel_3676/1137262478.py:48: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="60d", interval="60m", progress=False)
/tmp/ipykernel_3676/1137262478.py:48: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="60d", interval="60m", progress=False)
/tmp/ipykernel_3676/1137262478.py:48: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="60d", interval="60m", progress=False)
/tmp/ipykernel_3676/1137262478.py:48: FutureWarning: YF

             ESCANEO GLOBAL MULTIACTIVO - PLAN MAESTRO                    

🟢 TOTAL DE SEÑALES CALL DETECTADAS: 81

[SPY] 📅 2026-08-12 12:30:00-04:00 | Precio: 772.78 | SMA 40: 771.71 | Vol: 2,563,103 | 🎯 TP Limit: $4.500
[SPY] 📅 2026-08-12 13:30:00-04:00 | Precio: 773.09 | SMA 40: 771.76 | Vol: 2,417,715 | 🎯 TP Limit: $4.500
[SPY] 📅 2026-08-13 09:30:00-04:00 | Precio: 779.19 | SMA 40: 771.91 | Vol: 6,564,966 | 🎯 TP Limit: $4.500
[SPY] 📅 2026-08-14 14:30:00-04:00 | Precio: 776.16 | SMA 40: 773.92 | Vol: 3,082,814 | 🎯 TP Limit: $4.500
[DIA] 📅 2026-07-16 15:30:00-04:00 | Precio: 524.78 | SMA 40: 525.04 | Vol: 282,926 | 🎯 TP Limit: $4.500
[DIA] 📅 2026-07-17 09:30:00-04:00 | Precio: 524.56 | SMA 40: 525.03 | Vol: 2,027,512 | 🎯 TP Limit: $4.500
[DIA] 📅 2026-08-10 10:30:00-04:00 | Precio: 539.70 | SMA 40: 537.66 | Vol: 359,236 | 🎯 TP Limit: $4.500
[DIA] 📅 2026-08-10 15:30:00-04:00 | Precio: 539.02 | SMA 40: 539.12 | Vol: 332,359 | 🎯 TP Limit: $4.500
[AAPL] 📅 2026-07-20 13:30:00-04:00 | Preci

In [12]:
import pandas as pd
import yfinance as yf

universo_activos = [
    "SPY", "QQQ", "DIA",
    "AAPL", "AMZN", "GOOG", "MSFT", "NFLX",
    "CMG", "WMT", "COST", "MCD", "NKE",
    "JPM", "BAC", "WFC", "C",
    "JNJ", "KO", "PEP", "F"
]

print("Iniciando escaneo avanzado: Añadiendo Estrategia 5 (Caída + Hammer + Ruptura)...\n")

resultados_estrategia_5 = []

def calcular_take_profit(num_contratos=10, precio_contrato_inicial=0.30):
    comision_total = 7.00 + (1.25 * num_contratos)
    inversion_neta = (num_contratos * precio_contrato_inicial) + comision_total
    return (inversion_neta / num_contratos) * 2

for ticker in universo_activos:
    try:
        df = yf.download(ticker, period="60d", interval="60m", progress=False)
        if df.empty: continue
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        df.index = df.index.tz_convert('America/New_York')

        # 1. Detectar caída previa pronunciada (mínimo de los últimos 5 periodos menor al promedio)
        df["Caida_Previa"] = df["Close"] < df["Close"].shift(5)

        # 2. Identificar patrón Vela Hammer (Martillo)
        cuerpo = abs(df["Close"] - df["Open"])
        rango_total = df["High"] - df["Low"]
        sombra_inferior = df[["Open", "Close"]].min(axis=1) - df["Low"]
        sombra_superior = df["High"] - df[["Open", "Close"]].max(axis=1)

        # El cuerpo es pequeño, la sombra inferior es al menos el doble del cuerpo, y la superior es mínima
        df["Es_Hammer"] = (
            (cuerpo > 0) &
            (sombra_inferior >= (2 * cuerpo)) &
            (sombra_superior <= (0.5 * cuerpo)) &
            (rango_total > 0)
        )

        # 3. Gatillo: Ruptura alcista del máximo de la vela Hammer en la vela siguiente
        df["Hammer_Anterior"] = df["Es_Hammer"].shift(1)
        df["Ruptura_Hammer"] = df["High"] > df["High"].shift(1)
        df["Señal_Hammer_CALL"] = df["Caida_Previa"] & df["Hammer_Anterior"] & df["Ruptura_Hammer"]

        # Filtrar aciertos
        senales = df[df["Señal_Hammer_CALL"]]
        for idx, row in senales.iterrows():
            resultados_estrategia_5.append({
                "Ticker": ticker,
                "Fecha": idx,
                "Precio": row["Close"],
                "Volumen": row["Volume"],
                "TP": calcular_take_profit()
            })

    except Exception as e:
        continue

print("==========================================================================")
print("       RESULTADOS: ESTRATEGIA 5 (CAÍDA + HAMMER + RUPTURA CALL)           ")
print("==========================================================================")

if resultados_estrategia_5:
    print(f"\n🟢 TOTAL DE SEÑALES HAMMER DETECTADAS: {len(resultados_estrategia_5)}\n")
    for item in resultados_estrategia_5:
        print(f"[{item['Ticker']}] 📅 {item['Fecha']} | Precio: {item['Precio']:.2f} | Vol: {item['Volumen']:,} | 🎯 TP Limit: ${item['TP']:.3f}")
else:
    print("\n🟢 No hay patrones Hammer con ruptura en este tramo de fechas.")
print("==========================================================================")

Iniciando escaneo avanzado: Añadiendo Estrategia 5 (Caída + Hammer + Ruptura)...



/tmp/ipykernel_3676/2619868369.py:23: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="60d", interval="60m", progress=False)
/tmp/ipykernel_3676/2619868369.py:23: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="60d", interval="60m", progress=False)
/tmp/ipykernel_3676/2619868369.py:23: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="60d", interval="60m", progress=False)
/tmp/ipykernel_3676/2619868369.py:23: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="60d", interval="60m", progress=False)
/tmp/ipykernel_3676/2619868369.py:23: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="60d", interval="60m", progress=False)
/tmp/ipykernel_3676/2619868369.py:23: FutureWarning: YF

       RESULTADOS: ESTRATEGIA 5 (CAÍDA + HAMMER + RUPTURA CALL)           

🟢 TOTAL DE SEÑALES HAMMER DETECTADAS: 80

[SPY] 📅 2026-06-05 13:30:00-04:00 | Precio: 740.90 | Vol: 10,285,858 | 🎯 TP Limit: $4.500
[QQQ] 📅 2026-06-17 13:30:00-04:00 | Precio: 728.69 | Vol: 7,474,427 | 🎯 TP Limit: $4.500
[QQQ] 📅 2026-07-01 11:30:00-04:00 | Precio: 729.23 | Vol: 4,110,781 | 🎯 TP Limit: $4.500
[QQQ] 📅 2026-07-02 14:30:00-04:00 | Precio: 709.85 | Vol: 4,366,032 | 🎯 TP Limit: $4.500
[DIA] 📅 2026-06-08 14:30:00-04:00 | Precio: 508.93 | Vol: 456,689 | 🎯 TP Limit: $4.500
[DIA] 📅 2026-06-26 11:30:00-04:00 | Precio: 518.90 | Vol: 403,840 | 🎯 TP Limit: $4.500
[DIA] 📅 2026-07-07 14:30:00-04:00 | Precio: 527.84 | Vol: 267,546 | 🎯 TP Limit: $4.500
[DIA] 📅 2026-07-20 11:30:00-04:00 | Precio: 519.80 | Vol: 128,178 | 🎯 TP Limit: $4.500
[DIA] 📅 2026-07-22 09:30:00-04:00 | Precio: 522.39 | Vol: 1,287,492 | 🎯 TP Limit: $4.500
[DIA] 📅 2026-08-17 14:30:00-04:00 | Precio: 534.30 | Vol: 220,819 | 🎯 TP Limit: $4.500
[

In [13]:
import pandas as pd
import yfinance as yf

universo_activos = [
    "SPY", "QQQ", "DIA",
    "AAPL", "AMZN", "GOOG", "MSFT", "NFLX",
    "CMG", "WMT", "COST", "MCD", "NKE",
    "JPM", "BAC", "WFC", "C",
    "JNJ", "KO", "PEP", "F"
]

print("==========================================================================")
print("     ESCANEO MULTIACTIVO: ESTRATEGIA 1 & ESTRATEGIA 2 - PLAN MAESTRO      ")
print("==========================================================================")

resultados_est1 = []
resultados_est2 = []

def calcular_take_profit(num_contratos=10, precio_contrato_inicial=0.30):
    comision_total = 7.00 + (1.25 * num_contratos)
    inversion_neta = (num_contratos * precio_contrato_inicial) + comision_total
    return (inversion_neta / num_contratos) * 2

for ticker in universo_activos:
    try:
        df = yf.download(ticker, period="60d", interval="60m", progress=False)
        if df.empty: continue
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        df.index = df.index.tz_convert('America/New_York')

        # --- ESTRATEGIA 1: Ruptura de línea bajista sobre piso fuerte (Máxima Rentabilidad) ---
        df["Soporte_Fuerte"] = df["Low"].rolling(window=20).min()
        df["Max_Decreciente"] = (df["High"].shift(1) < df["High"].shift(2)) & (df["High"].shift(2) < df["High"].shift(3))
        df["Cerca_Soporte"] = (df["Low"] <= df["Soporte_Fuerte"] * 1.01)

        # Gatillo Estrategia 1
        df["Señal_Est_1"] = df["Max_Decreciente"] & (df["Close"] > df["Open"]) & (df["High"] > df["High"].shift(1)) & (df["Volume"] > df["Volume"].rolling(window=20).mean())

        # --- ESTRATEGIA 2: Ruptura de Rango / Máximo con Volumen Institucional ---
        df["Max_20"] = df["High"].rolling(window=20).max().shift(1)
        df["Volumen_Alto"] = df["Volume"] > (df["Volume"].rolling(window=20).mean() * 1.2)
        df["Señal_Est_2"] = (df["Close"] > df["Max_20"]) & df["Volumen_Alto"] & (df["Close"] > df["Open"])

        # Recolectar Estrategia 1
        est1_act = df[df["Señal_Est_1"]]
        for idx, row in est1_act.iterrows():
            resultados_est1.append({
                "Ticker": ticker, "Fecha": idx, "Precio": row["Close"],
                "Volumen": row["Volume"], "TP": calcular_take_profit()
            })

        # Recolectar Estrategia 2
        est2_act = df[df["Señal_Est_2"]]
        for idx, row in est2_act.iterrows():
            resultados_est2.append({
                "Ticker": ticker, "Fecha": idx, "Precio": row["Close"],
                "Volumen": row["Volume"], "TP": calcular_take_profit()
            })

    except Exception as e:
        continue

# --- REPORTES ---
print("\n--- [ESTRATEGIA 1] Ruptura sobre Piso Fuerte (Máxima Rentabilidad) ---")
if resultados_est1:
    print(f"Total señales Estrategia 1 detectadas: {len(resultados_est1)}\n")
    for item in resultados_est1:
        print(f"[{item['Ticker']}] 📅 {item['Fecha']} | Precio: {item['Precio']:.2f} | Vol: {item['Volumen']:,} | 🎯 TP Limit: ${item['TP']:.3f}")
else:
    print("No hay señales activas para la Estrategia 1 en este tramo.")

print("\n--- [ESTRATEGIA 2] Ruptura de Rango con Volumen Institucional ---")
if resultados_est2:
    print(f"Total señales Estrategia 2 detectadas: {len(resultados_est2)}\n")
    for item in resultados_est2:
        print(f"[{item['Ticker']}] 📅 {item['Fecha']} | Precio: {item['Precio']:.2f} | Vol: {item['Volumen']:,} | 🎯 TP Limit: ${item['TP']:.3f}")
else:
    print("No hay señales activas para la Estrategia 2 en este tramo.")

print("\n==========================================================================")
print("Escaneo de Estrategia 1 y Estrategia 2 completado 🚀")

     ESCANEO MULTIACTIVO: ESTRATEGIA 1 & ESTRATEGIA 2 - PLAN MAESTRO      


/tmp/ipykernel_3676/4136815385.py:26: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="60d", interval="60m", progress=False)
/tmp/ipykernel_3676/4136815385.py:26: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="60d", interval="60m", progress=False)
/tmp/ipykernel_3676/4136815385.py:26: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="60d", interval="60m", progress=False)
/tmp/ipykernel_3676/4136815385.py:26: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="60d", interval="60m", progress=False)
/tmp/ipykernel_3676/4136815385.py:26: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="60d", interval="60m", progress=False)
/tmp/ipykernel_3676/4136815385.py:26: FutureWarning: YF


--- [ESTRATEGIA 1] Ruptura sobre Piso Fuerte (Máxima Rentabilidad) ---
Total señales Estrategia 1 detectadas: 284

[SPY] 📅 2026-06-11 09:30:00-04:00 | Precio: 728.82 | Vol: 13,356,726 | 🎯 TP Limit: $4.500
[SPY] 📅 2026-06-17 09:30:00-04:00 | Precio: 751.69 | Vol: 8,522,283 | 🎯 TP Limit: $4.500
[SPY] 📅 2026-06-18 15:30:00-04:00 | Precio: 746.59 | Vol: 11,540,450 | 🎯 TP Limit: $4.500
[SPY] 📅 2026-06-24 09:30:00-04:00 | Precio: 736.49 | Vol: 9,730,442 | 🎯 TP Limit: $4.500
[SPY] 📅 2026-06-26 09:30:00-04:00 | Precio: 733.49 | Vol: 9,853,564 | 🎯 TP Limit: $4.500
[SPY] 📅 2026-07-02 09:30:00-04:00 | Precio: 747.94 | Vol: 8,848,490 | 🎯 TP Limit: $4.500
[SPY] 📅 2026-07-15 09:30:00-04:00 | Precio: 754.62 | Vol: 8,400,817 | 🎯 TP Limit: $4.500
[SPY] 📅 2026-07-16 15:30:00-04:00 | Precio: 750.76 | Vol: 11,103,382 | 🎯 TP Limit: $4.500
[SPY] 📅 2026-07-17 11:30:00-04:00 | Precio: 745.94 | Vol: 7,361,998 | 🎯 TP Limit: $4.500
[SPY] 📅 2026-07-23 15:30:00-04:00 | Precio: 738.17 | Vol: 11,577,395 | 🎯 TP Limi

In [14]:
import pandas as pd
import yfinance as yf
import warnings

# Silenciar advertencias secundarias de Pandas/Yfinance
warnings.simplefilter(action='ignore', category=FutureWarning)

universo_activos = [
    "SPY", "QQQ", "DIA",
    "AAPL", "AMZN", "GOOG", "MSFT", "NFLX",
    "CMG", "WMT", "COST", "MCD", "NKE",
    "JPM", "BAC", "WFC", "C",
    "JNJ", "KO", "PEP", "F"
]

print("==========================================================================")
print("      MATRIZ MAESTRA UNIFICADA - TODAS LAS ESTRATEGIAS ACTIVAS             ")
print("==========================================================================")

todas_las_senales = []

def calcular_take_profit(num_contratos=10, precio_contrato_inicial=0.30):
    comision_total = 7.00 + (1.25 * num_contratos)
    inversion_neta = (num_contratos * precio_contrato_inicial) + comision_total
    return (inversion_neta / num_contratos) * 2

for ticker in universo_activos:
    try:
        # Descarga limpia configurando auto_adjust
        df = yf.download(ticker, period="60d", interval="60m", progress=False, auto_adjust=False)
        if df.empty: continue
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        df.index = df.index.tz_convert('America/New_York')

        # Medias Móviles Base
        df["SMA_20"] = df["Close"].rolling(window=20).mean()
        df["SMA_40"] = df["Close"].rolling(window=40).mean()
        df["SMA_100"] = df["Close"].rolling(window=100).mean()
        df["SMA_200"] = df["Close"].rolling(window=200).mean()

        # 1. Pullback SMA 40 (CALL)
        tendencia_alcista = (df["SMA_20"] > df["SMA_40"]) & (df["SMA_40"] > df["SMA_100"]) & (df["SMA_100"] > df["SMA_200"])
        distancia_alcista = abs(df["Low"] - df["SMA_40"]) / df["SMA_40"]
        pullback_alcista = tendencia_alcista & (distancia_alcista <= 0.004)
        s_call = pullback_alcista.shift(1) & (df["Close"] > df["Open"]) & (df["High"] > df["High"].shift(1))

        # 2. Estrategia 1: Ruptura sobre piso fuerte
        soporte_fuerte = df["Low"].rolling(window=20).min()
        max_dec = (df["High"].shift(1) < df["High"].shift(2)) & (df["High"].shift(2) < df["High"].shift(3))
        s_est1 = max_dec & (df["Close"] > df["Open"]) & (df["High"] > df["High"].shift(1)) & (df["Volume"] > df["Volume"].rolling(window=20).mean())

        # 3. Estrategia 2: Ruptura de rango con volumen
        max_20 = df["High"].rolling(window=20).max().shift(1)
        s_est2 = (df["Close"] > max_20) & (df["Volume"] > (df["Volume"].rolling(window=20).mean() * 1.2)) & (df["Close"] > df["Open"])

        # 4. Estrategia 5: Patrón Hammer
        cuerpo = abs(df["Close"] - df["Open"])
        sombra_inf = df[["Open", "Close"]].min(axis=1) - df["Low"]
        sombra_sup = df["High"] - df[["Open", "Close"]].max(axis=1)
        es_hammer = (cuerpo > 0) & (sombra_inf >= (2 * cuerpo)) & (sombra_sup <= (0.5 * cuerpo))
        s_hammer = (df["Close"] < df["Close"].shift(5)) & es_hammer.shift(1) & (df["High"] > df["High"].shift(1))

        # Registrar cada tipo de señal encontrada
        for idx, row in df[s_call].iterrows():
            todas_las_senales.append({"Ticker": ticker, "Estrategia": "Pullback SMA40 (CALL)", "Fecha": idx, "Precio": round(row["Close"], 2), "TP Limit": round(calcular_take_profit(), 3)})
        for idx, row in df[s_est1].iterrows():
            todas_las_senales.append({"Ticker": ticker, "Estrategia": "Estrategia 1 (Piso Fuerte)", "Fecha": idx, "Precio": round(row["Close"], 2), "TP Limit": round(calcular_take_profit(), 3)})
        for idx, row in df[s_est2].iterrows():
            todas_las_senales.append({"Ticker": ticker, "Estrategia": "Estrategia 2 (Ruptura Rango)", "Fecha": idx, "Precio": round(row["Close"], 2), "TP Limit": round(calcular_take_profit(), 3)})
        for idx, row in df[s_hammer].iterrows():
            todas_las_senales.append({"Ticker": ticker, "Estrategia": "Estrategia 5 (Hammer)", "Fecha": idx, "Precio": round(row["Close"], 2), "TP Limit": round(calcular_take_profit(), 3)})

    except Exception as e:
        continue

# Generar y mostrar la tabla consolidada
df_final = pd.DataFrame(todas_las_senales)
if not df_final.empty:
    pd.set_option('display.max_rows', None)
    print(df_final.to_string(index=False))
    print(f"\n📊 TOTAL GENERAL DE SEÑALES UNIFICADAS: {len(df_final)}")
else:
    print("No se encontraron señales activas en este bloque.")
print("==========================================================================")

      MATRIZ MAESTRA UNIFICADA - TODAS LAS ESTRATEGIAS ACTIVAS             
Ticker                   Estrategia                     Fecha  Precio  TP Limit
   SPY        Pullback SMA40 (CALL) 2026-08-12 12:30:00-04:00  772.78       4.5
   SPY        Pullback SMA40 (CALL) 2026-08-12 13:30:00-04:00  773.09       4.5
   SPY        Pullback SMA40 (CALL) 2026-08-13 09:30:00-04:00  779.19       4.5
   SPY        Pullback SMA40 (CALL) 2026-08-14 14:30:00-04:00  776.16       4.5
   SPY   Estrategia 1 (Piso Fuerte) 2026-06-11 09:30:00-04:00  728.82       4.5
   SPY   Estrategia 1 (Piso Fuerte) 2026-06-17 09:30:00-04:00  751.69       4.5
   SPY   Estrategia 1 (Piso Fuerte) 2026-06-18 15:30:00-04:00  746.59       4.5
   SPY   Estrategia 1 (Piso Fuerte) 2026-06-24 09:30:00-04:00  736.49       4.5
   SPY   Estrategia 1 (Piso Fuerte) 2026-06-26 09:30:00-04:00  733.49       4.5
   SPY   Estrategia 1 (Piso Fuerte) 2026-07-02 09:30:00-04:00  747.94       4.5
   SPY   Estrategia 1 (Piso Fuerte) 2026-07-

In [15]:
# --- EXPORTAR Y RESUMIR LA MATRIZ MAESTRA ---

# 1. Agrupar por Ticker y Estrategia para ver el desglose exacto
resumen_estrategias = df_final.groupby(["Ticker", "Estrategia"]).size().reset_index(name="Total_Señales")

print("==================================================")
print("        RESUMEN INSTITUCIONAL POR ACTIVO          ")
print("==================================================")
print(resumen_estrategias.to_string(index=False))

# 2. Exportar automáticamente a un archivo CSV en tu Colab para abrir en Excel
nombre_archivo = "Plan_Maestro_Señales_Unificadas.csv"
df_final.to_csv(nombre_archivo, index=False)
print(f"\n📁 ¡Archivo exportado con éxito! Puedes descargarlo desde la pestaña de archivos de Colab: '{nombre_archivo}'")

        RESUMEN INSTITUCIONAL POR ACTIVO          
Ticker                   Estrategia  Total_Señales
  AAPL   Estrategia 1 (Piso Fuerte)             11
  AAPL Estrategia 2 (Ruptura Rango)             15
  AAPL        Estrategia 5 (Hammer)              3
  AAPL        Pullback SMA40 (CALL)              5
  AMZN   Estrategia 1 (Piso Fuerte)             11
  AMZN Estrategia 2 (Ruptura Rango)             10
  AMZN        Estrategia 5 (Hammer)              3
  AMZN        Pullback SMA40 (CALL)              4
   BAC   Estrategia 1 (Piso Fuerte)             20
   BAC Estrategia 2 (Ruptura Rango)             17
   BAC        Estrategia 5 (Hammer)              4
   BAC        Pullback SMA40 (CALL)             16
     C   Estrategia 1 (Piso Fuerte)             12
     C Estrategia 2 (Ruptura Rango)             14
     C        Estrategia 5 (Hammer)              5
   CMG   Estrategia 1 (Piso Fuerte)             12
   CMG Estrategia 2 (Ruptura Rango)             18
   CMG        Estrategia 5 (Ham

In [16]:
import pandas as pd
import yfinance as yf
import warnings
import requests

# Silenciar advertencias
warnings.simplefilter(action='ignore', category=FutureWarning)

# --- CONFIGURACIÓN DE TELEGRAM ---
TELEGRAM_TOKEN = "AAH5diJ9nkG7ySF4Z8AVdtS-8ZvsZYVv0pQ"       # Pega aquí el token de BotFather entre comillas
CHAT_ID = "8642681599"                 # ¡Tu Chat ID ya está configurado aquí!

def enviar_alerta_telegram(mensaje):
    """Función para enviar mensajes automáticos a tu Telegram"""
    url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
    payload = {
        "chat_id": CHAT_ID,
        "text": mensaje,
        "parse_mode": "Markdown"
    }
    try:
        response = requests.post(url, json=payload)
        return response.json()
    except Exception as e:
        print(f"Error al enviar Telegram: {e}")

universo_activos = [
    "SPY", "QQQ", "DIA",
    "AAPL", "AMZN", "GOOG", "MSFT", "NFLX",
    "CMG", "WMT", "COST", "MCD", "NKE",
    "JPM", "BAC", "WFC", "C",
    "JNJ", "KO", "PEP", "F"
]

print("==========================================================================")
print("     MATRIZ MAESTRA CON ALERTA AUTOMATIZADA DE TELEGRAM ACTIVA             ")
print("==========================================================================")

todas_las_senales = []

def calcular_take_profit(num_contratos=10, precio_contrato_inicial=0.30):
    comision_total = 7.00 + (1.25 * num_contratos)
    inversion_neta = (num_contratos * precio_contrato_inicial) + comision_total
    return (inversion_neta / num_contratos) * 2

for ticker in universo_activos:
    try:
        df = yf.download(ticker, period="60d", interval="60m", progress=False, auto_adjust=False)
        if df.empty: continue
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        df.index = df.index.tz_convert('America/New_York')

        # Medias Móviles Base
        df["SMA_20"] = df["Close"].rolling(window=20).mean()
        df["SMA_40"] = df["Close"].rolling(window=40).mean()
        df["SMA_100"] = df["Close"].rolling(window=100).mean()
        df["SMA_200"] = df["Close"].rolling(window=200).mean()

        # Estrategias
        tendencia_alcista = (df["SMA_20"] > df["SMA_40"]) & (df["SMA_40"] > df["SMA_100"]) & (df["SMA_100"] > df["SMA_200"])
        distancia_alcista = abs(df["Low"] - df["SMA_40"]) / df["SMA_40"]
        pullback_alcista = tendencia_alcista & (distancia_alcista <= 0.004)
        s_call = pullback_alcista.shift(1) & (df["Close"] > df["Open"]) & (df["High"] > df["High"].shift(1))

        soporte_fuerte = df["Low"].rolling(window=20).min()
        max_dec = (df["High"].shift(1) < df["High"].shift(2)) & (df["High"].shift(2) < df["High"].shift(3))
        s_est1 = max_dec & (df["Close"] > df["Open"]) & (df["High"] > df["High"].shift(1)) & (df["Volume"] > df["Volume"].rolling(window=20).mean())

        max_20 = df["High"].rolling(window=20).max().shift(1)
        s_est2 = (df["Close"] > max_20) & (df["Volume"] > (df["Volume"].rolling(window=20).mean() * 1.2)) & (df["Close"] > df["Open"])

        cuerpo = abs(df["Close"] - df["Open"])
        sombra_inf = df[["Open", "Close"]].min(axis=1) - df["Low"]
        sombra_sup = df["High"] - df[["Open", "Close"]].max(axis=1)
        es_hammer = (cuerpo > 0) & (sombra_inf >= (2 * cuerpo)) & (sombra_sup <= (0.5 * cuerpo))
        s_hammer = (df["Close"] < df["Close"].shift(5)) & es_hammer.shift(1) & (df["High"] > df["High"].shift(1))

        estrategias_activas = [
            (s_call, "Pullback SMA40 (CALL)"),
            (s_est1, "Estrategia 1 (Piso Fuerte)"),
            (s_est2, "Estrategia 2 (Ruptura Rango)"),
            (s_hammer, "Estrategia 5 (Hammer)")
        ]

        for condicion, nombre_est in estrategias_activas:
            for idx, row in df[condicion].iterrows():
                tp_val = round(calcular_take_profit(), 3)
                precio_val = round(row["Close"], 2)
                todas_las_senales.append({"Ticker": ticker, "Estrategia": nombre_est, "Fecha": idx, "Precio": precio_val, "TP Limit": tp_val})

    except Exception as e:
        continue

df_final = pd.DataFrame(todas_las_senales)
if not df_final.empty:
    print(f"\n📊 TOTAL DE SEÑALES ENCONTRADAS: {len(df_final)}")
    ultimas_senales = df_final.tail(5)

    mensaje_telegram = "🚨 *NUEVAS SEÑALES DE TRADING DETECTADAS* 🚨\n\n"
    for _, r in ultimas_senales.iterrows():
        mensaje_telegram += f"📌 *{r['Ticker']}* | {r['Estrategia']}\n" \
                            f"📅 {r['Fecha']}\n" \
                            f"💵 Precio: `${r['Precio']}` | 🎯 TP: `${r['TP Limit']}`\n\n"

    respuesta = enviar_alerta_telegram(mensaje_telegram)
    print("📲 ¡Alerta enviada a Telegram con éxito!")
else:
    print("No se encontraron señales activas.")
print("==========================================================================")

     MATRIZ MAESTRA CON ALERTA AUTOMATIZADA DE TELEGRAM ACTIVA             

📊 TOTAL DE SEÑALES ENCONTRADAS: 710
📲 ¡Alerta enviada a Telegram con éxito!


In [17]:
import requests

TELEGRAM_TOKEN = "8556809936:AAH5diJ9nkG7ySF4Z8AVdt

S-8ZvsZYVv0pQ"  # Pon tu token aquí
CHAT_ID = "8642681599"

url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
payload = {
    "chat_id": CHAT_ID,
    "text": "Hola Franz, esto es una prueba directa desde Colab 🚀",
}

response = requests.post(url, json=payload)
print(response.json())

SyntaxError: unterminated string literal (detected at line 3) (1024194122.py, line 3)

In [ ]:
import requests

TELEGRAM_TOKEN = "8556809936:AAH5diJ9nkG7ySF4Z8AVdtS-8ZvsZYVv0pQ"
CHAT_ID = "8642681599"

url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
payload = {
    "chat_id": CHAT_ID,
    "text": "Hola Franz, prueba directa desde Colab 🚀 ¡Sistema de trading conectado!",
}

response = requests.post(url, json=payload)
print(response.json())

In [ ]:
import requests

TELEGRAM_TOKEN = "8556809936:AAH5diJ9nkG7ySF4Z8AVdtS-8ZvsZYVv0pQ"
CHAT_ID = "8642681599"

url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
payload = {
    "chat_id": CHAT_ID,
    "text": "Hola Franz, prueba directa desde Colab 🚀 ¡Sistema conectado!",
}

response = requests.post(url, json=payload)
print(response.json())

In [ ]:
import requests

TELEGRAM_TOKEN = "TU_TOKEN_AQUI"  # Pon tu token aquí
CHAT_ID = "8642681599"

url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
payload = {
    "chat_id": CHAT_ID,
    "text": "Hola Franz, esto es una prueba directa desde Colab 🚀",
}

response = requests.post(url, json=payload)
print(response.json())

In [ ]:
import requests

TELEGRAM_TOKEN = "8556809936:AAH5diJ9nkG7ySF4Z8AVdtS-8ZvsZYVv0pQ"
CHAT_ID = "8642681599"

url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
payload = {
    "chat_id": CHAT_ID,
    "text": "Hola Franz, prueba directa desde Colab 🚀 ¡Sistema conectado!",
}

response = requests.post(url, json=payload)
print(response.json())

In [ ]:
import requests

token_part1 = "8556809936:AAH5diJ9nkG7ySF4Z8AVdt"
token_part2 = "S-8ZvsZYVv0pQ"
TELEGRAM_TOKEN = token_part1 + token_part2

CHAT_ID = "8642681599"

url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
payload = {
    "chat_id": CHAT_ID,
    "text": "Hola Franz, prueba directa desde Colab 🚀 ¡Sistema conectado!",
}

response = requests.post(url, json=payload)
print(response.json())

In [ ]:
import requests

# Unimos el token limpiamente en una sola línea mediante código
p1 = "8556809936:AAH5diJ9nkG7ySF4Z8AVdt"
p2 = "S-8ZvsZYVv0pQ"
TELEGRAM_TOKEN = p1 + p2

CHAT_ID = "8642681599"

url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
payload = {
    "chat_id": CHAT_ID,
    "text": "Hola Franz 🚀 ¡Prueba final conectada con éxito!",
}

response = requests.post(url, json=payload)
print(response.json())

In [ ]:
import requests

# Unimos tu nuevo token limpio en dos partes para evitar saltos de línea
p1 = "8556809936:AAFaziJLF1BOgnyXSBfJgt6S"
p2 = "J2asida9daE"
TELEGRAM_TOKEN = p1 + p2

CHAT_ID = "8642681599"

url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
payload = {
    "chat_id": CHAT_ID,
    "text": "Hola Franz 🚀 ¡Prueba con el nuevo token definitivo!",
}

response = requests.post(url, json=payload)
print(response.json())

In [ ]:
import pandas as pd
import yfinance as yf
import warnings
import requests

# Silenciar advertencias
warnings.simplefilter(action='ignore', category=FutureWarning)

# --- CONFIGURACIÓN DE TELEGRAM (CON TOKEN LIMPIO) ---
p1 = "8556809936:AAFaziJLF1BOgnyXSBfJgt6S"
p2 = "J2asida9daE"
TELEGRAM_TOKEN = p1 + p2
CHAT_ID = "8642681599"

def enviar_alerta_telegram(mensaje):
    """Función para enviar mensajes automáticos a tu Telegram"""
    url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
    payload = {
        "chat_id": CHAT_ID,
        "text": mensaje,
        "parse_mode": "Markdown"
    }
    try:
        response = requests.post(url, json=payload)
        return response.json()
    except Exception as e:
        print(f"Error al enviar Telegram: {e}")

universo_activos = [
    "SPY", "QQQ", "DIA",
    "AAPL", "AMZN", "GOOG", "MSFT", "NFLX",
    "CMG", "WMT", "COST", "MCD", "NKE",
    "JPM", "BAC", "WFC", "C",
    "JNJ", "KO", "PEP", "F"
]

print("==========================================================================")
print("     MATRIZ MAESTRA CON ALERTA AUTOMATIZADA DE TELEGRAM ACTIVA             ")
print("==========================================================================")

todas_las_senales = []

def calcular_take_profit(num_contratos=10, precio_contrato_inicial=0.30):
    comision_total = 7.00 + (1.25 * num_contratos)
    inversion_neta = (num_contratos * precio_contrato_inicial) + comision_total
    return (inversion_neta / num_contratos) * 2

for ticker in universo_activos:
    try:
        df = yf.download(ticker, period="60d", interval="60m", progress=False, auto_adjust=False)
        if df.empty: continue
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        df.index = df.index.tz_convert('America/New_York')

        # Medias Móviles Base
        df["SMA_20"] = df["Close"].rolling(window=20).mean()
        df["SMA_40"] = df["Close"].rolling(window=40).mean()
        df["SMA_100"] = df["Close"].rolling(window=100).mean()
        df["SMA_200"] = df["Close"].rolling(window=200).mean()

        # Estrategias
        tendencia_alcista = (df["SMA_20"] > df["SMA_40"]) & (df["SMA_40"] > df["SMA_100"]) & (df["SMA_100"] > df["SMA_200"])
        distancia_alcista = abs(df["Low"] - df["SMA_40"]) / df["SMA_40"]
        pullback_alcista = tendencia_alcista & (distancia_alcista <= 0.004)
        s_call = pullback_alcista.shift(1) & (df["Close"] > df["Open"]) & (df["High"] > df["High"].shift(1))

        soporte_fuerte = df["Low"].rolling(window=20).min()
        max_dec = (df["High"].shift(1) < df["High"].shift(2)) & (df["High"].shift(2) < df["High"].shift(3))
        s_est1 = max_dec & (df["Close"] > df["Open"]) & (df["High"] > df["High"].shift(1)) & (df["Volume"] > df["Volume"].rolling(window=20).mean())

        max_20 = df["High"].rolling(window=20).max().shift(1)
        s_est2 = (df["Close"] > max_20) & (df["Volume"] > (df["Volume"].rolling(window=20).mean() * 1.2)) & (df["Close"] > df["Open"])

        cuerpo = abs(df["Close"] - df["Open"])
        sombra_inf = df[["Open", "Close"]].min(axis=1) - df["Low"]
        sombra_sup = df["High"] - df[["Open", "Close"]].max(axis=1)
        es_hammer = (cuerpo > 0) & (sombra_inf >= (2 * cuerpo)) & (sombra_sup <= (0.5 * cuerpo))
        s_hammer = (df["Close"] < df["Close"].shift(5)) & es_hammer.shift(1) & (df["High"] > df["High"].shift(1))

        estrategias_activas = [
            (s_call, "Pullback SMA40 (CALL)"),
            (s_est1, "Estrategia 1 (Piso Fuerte)"),
            (s_est2, "Estrategia 2 (Ruptura Rango)"),
            (s_hammer, "Estrategia 5 (Hammer)")
        ]

        for condicion, nombre_est in estrategias_activas:
            for idx, row in df[condicion].iterrows():
                tp_val = round(calcular_take_profit(), 3)
                precio_val = round(row["Close"], 2)
                todas_las_senales.append({"Ticker": ticker, "Estrategia": nombre_est, "Fecha": idx, "Precio": precio_val, "TP Limit": tp_val})

    except Exception as e:
        continue

df_final = pd.DataFrame(todas_las_senales)
if not df_final.empty:
    print(f"\n📊 TOTAL DE SEÑALES ENCONTRADAS: {len(df_final)}")
    ultimas_senales = df_final.tail(5)

    mensaje_telegram = "🚨 *NUEVAS SEÑALES DE TRADING DETECTADAS* 🚨\n\n"
    for _, r in ultimas_senales.iterrows():
        mensaje_telegram += f"📌 *{r['Ticker']}* | {r['Estrategia']}\n" \
                            f"📅 {r['Fecha']}\n" \
                            f"💵 Precio: `${r['Precio']}` | 🎯 TP: `${r['TP Limit']}`\n\n"

    respuesta = enviar_alerta_telegram(mensaje_telegram)
    print("📲 ¡Alerta enviada a Telegram con éxito!")
else:
    print("No se encontraron señales activas.")
print("==========================================================================")


In [ ]:
import pandas as pd
import yfinance as yf
import warnings
import requests

# Silenciar advertencias
warnings.simplefilter(action='ignore', category=FutureWarning)

# --- CONFIGURACIÓN DE TELEGRAM ---
p1 = "8556809936:AAFaziJLF1BOgnyXSBfJgt6S"
p2 = "J2asida9daE"
TELEGRAM_TOKEN = p1 + p2
CHAT_ID = "8642681599"

# Diccionario para traducir los códigos raros a nombres fáciles
nombres_amigables = {
    "SPY": "S&P 500 (Las 500 empresas más grandes de EE.UU.)",
    "QQQ": "Invesco QQQ (Las 100 mayores empresas de tecnología, como Apple y Microsoft)",
    "DIA": "Dow Jones (Las 30 empresas industriales más famosas de EE.UU.)",
    "AAPL": "Apple (Fabricantes del iPhone)",
    "AMZN": "Amazon (La tienda gigante de internet)",
    "GOOG": "Google (El buscador y YouTube)",
    "MSFT": "Microsoft (Dueños de Windows y Xbox)",
    "NFLX": "Netflix (Películas y series en streaming)",
    "CMG": "Chipotle (Restaurantes de comida)",
    "WMT": "Walmart (Supermercados gigantes)",
    "COST": "Costco (Tiendas grandes al por mayor)",
    "MCD": "McDonald's (Hamburguesas)",
    "NKE": "Nike (Zapatillas y ropa deportiva)",
    "JPM": "JPMorgan Chase (Banco muy grande)",
    "BAC": "Bank of America (Banco de América)",
    "WFC": "Wells Fargo (Banco americano)",
    "C": "Citigroup (Banco global)",
    "JNJ": "Johnson & Johnson (Productos de salud y medicinas)",
    "KO": "Coca-Cola (El famoso refresco)",
    "PEP": "PepsiCo (Pepsi y snacks como Lay's)",
    "F": "Ford (La fábrica de coches)"
}

# Explicaciones sencillas para niños de cada estrategia
explicaciones_estrategias = {
    "Pullback SMA40 (CALL)": "El precio bajó un poquito para tomar carrerilla y ahora parece que va a volver a subir con fuerza.",
    "Estrategia 1 (Piso Fuerte)": "La acción tocó un suelo muy sólido donde los compradores dijeron '¡de aquí no pasa!' y empezaron a comprar.",
    "Estrategia 2 (Ruptura Rango)": "Rompió su récord de precio anterior con mucha gente comprando a la vez. ¡Va con ganas de subir más!",
    "Estrategia 5 (Hammer)": "Hizo forma de 'martillo': los vendedores intentaron tirar el precio abajo pero los compradores barrieron y lo subieron."
}

def enviar_alerta_telegram(mensaje):
    url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
    payload = {
        "chat_id": CHAT_ID,
        "text": mensaje,
        "parse_mode": "Markdown"
    }
    try:
        response = requests.post(url, json=payload)
        return response.json()
    except Exception as e:
        print(f"Error al enviar Telegram: {e}")

universo_activos = list(nombres_amigables.keys())

print("==========================================================================")
print("     MATRIZ MAESTRA - EXPLICACIONES PARA TODOS (MODO SENCILLO)            ")
print("==========================================================================")

todas_las_senales = []

def calcular_take_profit(num_contratos=10, precio_contrato_inicial=0.30):
    comision_total = 7.00 + (1.25 * num_contratos)
    inversion_neta = (num_contratos * precio_contrato_inicial) + comision_total
    return (inversion_neta / num_contratos) * 2

for ticker in universo_activos:
    try:
        df = yf.download(ticker, period="60d", interval="60m", progress=False, auto_adjust=False)
        if df.empty: continue
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        df.index = df.index.tz_convert('America/New_York')

        # Medias Móviles Base
        df["SMA_20"] = df["Close"].rolling(window=20).mean()
        df["SMA_40"] = df["Close"].rolling(window=40).mean()
        df["SMA_100"] = df["Close"].rolling(window=100).mean()
        df["SMA_200"] = df["Close"].rolling(window=200).mean()

        # Estrategias
        tendencia_alcista = (df["SMA_20"] > df["SMA_40"]) & (df["SMA_40"] > df["SMA_100"]) & (df["SMA_100"] > df["SMA_200"])
        distancia_alcista = abs(df["Low"] - df["SMA_40"]) / df["SMA_40"]
        pullback_alcista = tendencia_alcista & (distancia_alcista <= 0.004)
        s_call = pullback_alcista.shift(1) & (df["Close"] > df["Open"]) & (df["High"] > df["High"].shift(1))

        max_dec = (df["High"].shift(1) < df["High"].shift(2)) & (df["High"].shift(2) < df["High"].shift(3))
        s_est1 = max_dec & (df["Close"] > df["Open"]) & (df["High"] > df["High"].shift(1)) & (df["Volume"] > df["Volume"].rolling(window=20).mean())

        max_20 = df["High"].rolling(window=20).max().shift(1)
        s_est2 = (df["Close"] > max_20) & (df["Volume"] > (df["Volume"].rolling(window=20).mean() * 1.2)) & (df["Close"] > df["Open"])

        cuerpo = abs(df["Close"] - df["Open"])
        sombra_inf = df[["Open", "Close"]].min(axis=1) - df["Low"]
        sombra_sup = df["High"] - df[["Open", "Close"]].max(axis=1)
        es_hammer = (cuerpo > 0) & (sombra_inf >= (2 * cuerpo)) & (sombra_sup <= (0.5 * cuerpo))
        s_hammer = (df["Close"] < df["Close"].shift(5)) & es_hammer.shift(1) & (df["High"] > df["High"].shift(1))

        estrategias_activas = [
            (s_call, "Pullback SMA40 (CALL)"),
            (s_est1, "Estrategia 1 (Piso Fuerte)"),
            (s_est2, "Estrategia 2 (Ruptura Rango)"),
            (s_hammer, "Estrategia 5 (Hammer)")
        ]

        for condicion, nombre_est in estrategias_activas:
            for idx, row in df[condicion].iterrows():
                tp_val = round(calcular_take_profit(), 3)
                precio_val = round(row["Close"], 2)
                todas_las_senales.append({"Ticker": ticker, "Estrategia": nombre_est, "Fecha": idx, "Precio": precio_val, "TP Limit": tp_val})

    except Exception as e:
        continue

df_final = pd.DataFrame(todas_las_senales)
if not df_final.empty:
    print(f"\n📊 TOTAL DE SEÑALES ENCONTRADAS: {len(df_final)}")
    ultimas_senales = df_final.tail(3) # Cogemos las 3 últimas para que el mensaje no sea gigante

    mensaje_telegram = "🎯 *¡AVISO SENCILLO DE INVERSIÓN!* 🎯\n\n"
    for _, r in ultimas_senales.iterrows():
        t = r['Ticker']
        nombre_largo = nombres_amigables.get(t, t)
        est = r['Estrategia']
        explicacion = explicaciones_estrategias.get(est, "Movimiento interesante detectado.")

        mensaje_telegram += f"🏢 *Empresa:* {nombre_largo}\n" \
                            f"📈 *Qué pasó:* {explicacion}\n" \
                            f"📅 *Cuándo:* {r['Fecha']}\n" \
                            f"💵 *Precio actual:* `${r['Precio']}`\n" \
                            f"🎯 *Objetivo de ganancia (TP):* `${r['TP Limit']}`\n\n" \
                            f"-----------------------------------\n\n"

    respuesta = enviar_alerta_telegram(mensaje_telegram)
    print("📲 ¡Alerta súper explicada enviada a Telegram con éxito!")
else:
    print("No se encontraron señales activas.")
print("==========================================================================")

In [ ]:
import pandas as pd
import yfinance as yf
import warnings
import requests

# Silenciar advertencias
warnings.simplefilter(action='ignore', category=FutureWarning)

# --- CONFIGURACIÓN DE TELEGRAM ---
p1 = "8556809936:AAFaziJLF1BOgnyXSBfJgt6S"
p2 = "J2asida9daE"
TELEGRAM_TOKEN = p1 + p2
CHAT_ID = "8642681599"

# Nombres amigables para las empresas
nombres_amigables = {
    "SPY": "S&P 500 (SPY)",
    "QQQ": "Invesco QQQ (Tecnología)",
    "DIA": "Dow Jones (DIA)",
    "AAPL": "Apple",
    "AMZN": "Amazon",
    "GOOG": "Google",
    "MSFT": "Microsoft",
    "NFLX": "Netflix",
    "CMG": "Chipotle",
    "WMT": "Walmart",
    "COST": "Costco",
    "MCD": "McDonald's",
    "NKE": "Nike",
    "JPM": "JPMorgan Chase",
    "BAC": "Bank of America",
    "WFC": "Wells Fargo",
    "C": "Citigroup",
    "JNJ": "Johnson & Johnson",
    "KO": "Coca-Cola",
    "PEP": "PepsiCo",
    "F": "Ford"
}

def enviar_alerta_telegram(mensaje):
    url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
    payload = {
        "chat_id": CHAT_ID,
        "text": mensaje,
        "parse_mode": "Markdown"
    }
    try:
        response = requests.post(url, json=payload)
        return response.json()
    except Exception as e:
        print(f"Error al enviar Telegram: {e}")

universo_activos = list(nombres_amigables.keys())

print("==========================================================================")
print("     MATRIZ MAESTRA - FORMATO SÚPER SENCILLO PARA FRAN                     ")
print("==========================================================================")

todas_las_senales = []

for ticker in universo_activos:
    try:
        df = yf.download(ticker, period="60d", interval="60m", progress=False, auto_adjust=False)
        if df.empty: continue
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        df.index = df.index.tz_convert('America/New_York')

        # Medias Móviles Base
        df["SMA_20"] = df["Close"].rolling(window=20).mean()
        df["SMA_40"] = df["Close"].rolling(window=40).mean()
        df["SMA_100"] = df["Close"].rolling(window=100).mean()
        df["SMA_200"] = df["Close"].rolling(window=200).mean()

        # Estrategias
        tendencia_alcista = (df["SMA_20"] > df["SMA_40"]) & (df["SMA_40"] > df["SMA_100"]) & (df["SMA_100"] > df["SMA_200"])
        distancia_alcista = abs(df["Low"] - df["SMA_40"]) / df["SMA_40"]
        pullback_alcista = tendencia_alcista & (distancia_alcista <= 0.004)
        s_call = pullback_alcista.shift(1) & (df["Close"] > df["Open"]) & (df["High"] > df["High"].shift(1))

        max_dec = (df["High"].shift(1) < df["High"].shift(2)) & (df["High"].shift(2) < df["High"].shift(3))
        s_est1 = max_dec & (df["Close"] > df["Open"]) & (df["High"] > df["High"].shift(1)) & (df["Volume"] > df["Volume"].rolling(window=20).mean())

        max_20 = df["High"].rolling(window=20).max().shift(1)
        s_est2 = (df["Close"] > max_20) & (df["Volume"] > (df["Volume"].rolling(window=20).mean() * 1.2)) & (df["Close"] > df["Open"])

        cuerpo = abs(df["Close"] - df["Open"])
        sombra_inf = df[["Open", "Close"]].min(axis=1) - df["Low"]
        sombra_sup = df["High"] - df[["Open", "Close"]].max(axis=1)
        es_hammer = (cuerpo > 0) & (sombra_inf >= (2 * cuerpo)) & (sombra_sup <= (0.5 * cuerpo))
        s_hammer = (df["Close"] < df["Close"].shift(5)) & es_hammer.shift(1) & (df["High"] > df["High"].shift(1))

        estrategias_activas = [
            (s_call, "Pullback SMA40 (CALL)"),
            (s_est1, "Estrategia 1 (Piso Fuerte)"),
            (s_est2, "Estrategia 2 (Ruptura Rango)"),
            (s_hammer, "Estrategia 5 (Hammer)")
        ]

        for condicion, nombre_est in estrategias_activas:
            for idx, row in df[condicion].iterrows():
                precio_val = round(row["Close"], 2)
                todas_las_senales.append({"Ticker": ticker, "Estrategia": nombre_est, "Fecha": idx, "Precio": precio_val})

    except Exception as e:
        continue

df_final = pd.DataFrame(todas_las_senales)
if not df_final.empty:
    print(f"\n📊 TOTAL DE SEÑALES ENCONTRADAS: {len(df_final)}")
    ultimas_senales = df_final.tail(3) # Cogemos las 3 últimas para que sea cómodo de leer

    mensaje_telegram = "🚨 *¡NUEVA ALERTA PARA TI!* 🚨\n\n"
    for _, r in ultimas_senales.iterrows():
        nombre_empresa = nombres_amigables.get(r['Ticker'], r['Ticker'])
        estrategia = r['Estrategia']
        fecha = r['Fecha']
        precio = r['Precio']

        mensaje_telegram += f"Fran, hemos detectado que puedes invertir con la estrategia *{estrategia}* en *{nombre_empresa}*, ¿vale? Y puedes comprar call a partir de esta vela del *{fecha}* (Precio: `${precio}`).\n\n" \
                            f"-----------------------------------\n\n"

    respuesta = enviar_alerta_telegram(mensaje_telegram)
    print("📲 ¡Alerta clara y sencilla enviada a Telegram!")
else:
    print("No se encontraron señales activas.")
print("==========================================================================")

In [ ]:
import pandas as pd
import yfinance as yf
import warnings
import requests

# Silenciar advertencias
warnings.simplefilter(action='ignore', category=FutureWarning)

# --- CONFIGURACIÓN DE TELEGRAM ---
p1 = "8556809936:AAFaziJLF1BOgnyXSBfJgt6S"
p2 = "J2asida9daE"
TELEGRAM_TOKEN = p1 + p2
CHAT_ID = "8642681599"

# Nombres amigables para las empresas y ETFs
nombres_amigables = {
    "SPY": "S&P 500 (SPY)",
    "QQQ": "Invesco QQQ (Tecnología)",
    "DIA": "Dow Jones (DIA)",
    "AAPL": "Apple",
    "AMZN": "Amazon",
    "GOOG": "Google",
    "MSFT": "Microsoft",
    "NFLX": "Netflix",
    "CMG": "Chipotle",
    "WMT": "Walmart",
    "COST": "Costco",
    "MCD": "McDonald's",
    "NKE": "Nike",
    "JPM": "JPMorgan Chase",
    "BAC": "Bank of America",
    "WFC": "Wells Fargo",
    "C": "Citigroup",
    "JNJ": "Johnson & Johnson",
    "KO": "Coca-Cola",
    "PEP": "PepsiCo",
    "F": "Ford"
}

def enviar_alerta_telegram(mensaje):
    url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
    payload = {
        "chat_id": CHAT_ID,
        "text": mensaje,
        "parse_mode": "Markdown"
    }
    try:
        response = requests.post(url, json=payload)
        return response.json()
    except Exception as e:
        print(f"Error al enviar Telegram: {e}")

universo_activos = list(nombres_amigables.keys())

print("==========================================================================")
print("     MONITOR EN TIEMPO REAL - CALL / PUT & VELAS HORA A HORA              ")
print("==========================================================================")

alertas_para_enviar = []

for ticker in universo_activos:
    try:
        # Descargamos datos recientes de gráficos de 1 hora
        df = yf.download(ticker, period="5d", interval="60m", progress=False, auto_adjust=False)
        if df.empty: continue
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        df.index = df.index.tz_convert('America/New_York')

        # Analizamos la última vela cerrada
        ultima_vela = df.iloc[-1]
        hora_vela = ultima_vela.name.strftime('%H:%M')
        fecha_vela = ultima_vela.name.strftime('%Y-%m-%d')

        open_price = ultima_vela["Open"]
        close_price = ultima_vela["Close"]
        precio_actual = round(close_price, 2)

        # Determinamos si es Alcista (Verde -> Call) o Bajista (Roja -> Put)
        if close_price > open_price:
            tipo_operacion = "CALL 🟢 (Alcista / Verde)"
            instruccion = "puedes buscar una entrada en compra (Call)"
        else:
            tipo_operacion = "PUT 🔴 (Bajista / Roja)"
            instruccion = "puedes buscar una entrada en venta (Put)"

        nombre_empresa = nombres_amigables.get(ticker, ticker)

        alertas_para_enviar.append({
            "Empresa": nombre_empresa,
            "Hora": hora_vela,
            "Fecha": fecha_vela,
            "Operacion": tipo_operacion,
            "Instruccion": instruccion,
            "Precio": precio_actual
        })

    except Exception as e:
        continue

if alertas_para_enviar:
    # Seleccionamos algunas de las principales para no saturar el chat
    mensaje_telegram = "🔔 *REPORTE DE APERTURA Y VELAS* 🔔\n\n"
    mensaje_telegram += "Fran, la bolsa abre a las 9:30 y aquí tienes el escaneo de las últimas velas:\n\n"

    # Tomamos 4 activos de muestra para que el mensaje sea limpio y directo
    for item in alertas_para_enviar[:4]:
        mensaje_telegram += f"📌 *{item['Empresa']}* ({item['Fecha']} a las {item['Hora']}):\n" \
                            f"La vela fue de tipo *{item['Operacion']}* a ${item['Precio']}.\n" \
                            f"➡️ Fran, {item['Instruccion']}.\n\n" \
                            f"-----------------------------------\n\n"

    respuesta = enviar_alerta_telegram(mensaje_telegram)
    print("📲 ¡Reporte de velas y Call/Put enviado a Telegram con éxito!")
else:
    print("No se pudieron generar alertas en este momento.")
print("==========================================================================")

In [ ]:
import pandas as pd
import yfinance as yf
import warnings
import requests

# Silenciar advertencias
warnings.simplefilter(action='ignore', category=FutureWarning)

# --- CONFIGURACIÓN DE TELEGRAM ---
p1 = "8556809936:AAFaziJLF1BOgnyXSBfJgt6S"
p2 = "J2asida9daE"
TELEGRAM_TOKEN = p1 + p2
CHAT_ID = "8642681599"

# Nombres amigables para las empresas y ETFs
nombres_amigables = {
    "SPY": "S&P 500 (SPY)",
    "QQQ": "Invesco QQQ (Tecnología)",
    "DIA": "Dow Jones (DIA)",
    "AAPL": "Apple",
    "AMZN": "Amazon",
    "GOOG": "Google",
    "MSFT": "Microsoft",
    "NFLX": "Netflix",
    "CMG": "Chipotle",
    "WMT": "Walmart",
    "COST": "Costco",
    "MCD": "McDonald's",
    "NKE": "Nike",
    "JPM": "JPMorgan Chase",
    "BAC": "Bank of America",
    "WFC": "Wells Fargo",
    "C": "Citigroup",
    "JNJ": "Johnson & Johnson",
    "KO": "Coca-Cola",
    "PEP": "PepsiCo",
    "F": "Ford"
}

def enviar_alerta_telegram(mensaje):
    url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
    payload = {
        "chat_id": CHAT_ID,
        "text": mensaje,
        "parse_mode": "Markdown"
    }
    try:
        response = requests.post(url, json=payload)
        return response.json()
    except Exception as e:
        print(f"Error al enviar Telegram: {e}")

universo_activos = list(nombres_amigables.keys())

print("==========================================================================")
print("     MATRIZ INTELIGENTE - DETECCIÓN AUTOMÁTICA DE CALL / PUT               ")
print("==========================================================================")

todas_las_senales = []

for ticker in universo_activos:
    try:
        # Descargamos datos de gráficos de 1 hora
        df = yf.download(ticker, period="30d", interval="60m", progress=False, auto_adjust=False)
        if df.empty: continue
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        df.index = df.index.tz_convert('America/New_York')

        # Medias Móviles Base
        df["SMA_20"] = df["Close"].rolling(window=20).mean()
        df["SMA_40"] = df["Close"].rolling(window=40).mean()
        df["SMA_100"] = df["Close"].rolling(window=100).mean()
        df["SMA_200"] = df["Close"].rolling(window=200).mean()

        # --- ESTRATEGIAS ALCISTAS (CALL) ---
        tendencia_alcista = (df["SMA_20"] > df["SMA_40"]) & (df["SMA_40"] > df["SMA_100"])
        distancia_alcista = abs(df["Low"] - df["SMA_40"]) / df["SMA_40"]
        pullback_alcista = tendencia_alcista & (distancia_alcista <= 0.005)
        s_call = pullback_alcista.shift(1) & (df["Close"] > df["Open"]) & (df["High"] > df["High"].shift(1))

        max_20 = df["High"].rolling(window=20).max().shift(1)
        s_ruptura_call = (df["Close"] > max_20) & (df["Volume"] > (df["Volume"].rolling(window=20).mean() * 1.2)) & (df["Close"] > df["Open"])

        # --- ESTRATEGIAS BAJISTAS (PUT) ---
        tendencia_bajista = (df["SMA_20"] < df["SMA_40"]) & (df["SMA_40"] < df["SMA_100"])
        min_20 = df["Low"].rolling(window=20).min().shift(1)
        s_ruptura_put = (df["Close"] < min_20) & (df["Volume"] > (df["Volume"].rolling(window=20).mean() * 1.2)) & (df["Close"] < df["Open"])

        # Evaluamos si alguna condición se cumple en el DataFrame
        estrategias_a_evaluar = [
            (s_call, "Pullback SMA40", "CALL 🟢 (Compra alcista)"),
            (s_ruptura_call, "Ruptura de Rango", "CALL 🟢 (Ruptura al alza)"),
            (s_ruptura_put, "Ruptura de Soporte", "PUT 🔴 (Caída bajista)")
        ]

        for condicion, nombre_est, tipo_op in estrategias_a_evaluar:
            for idx, row in df[condicion].iterrows():
                precio_val = round(row["Close"], 2)
                fecha_str = idx.strftime('%Y-%m-%d %H:%M')
                todas_las_senales.append({
                    "Ticker": ticker,
                    "Estrategia": nombre_est,
                    "Operacion": tipo_op,
                    "Fecha": fecha_str,
                    "Precio": precio_val
                })

    except Exception as e:
        continue

df_final = pd.DataFrame(todas_las_senales)
if not df_final.empty:
    print(f"\n📊 TOTAL DE SEÑALES ENCONTRADAS: {len(df_final)}")
    ultimas_senales = df_final.tail(3) # Las 3 últimas detectadas

    mensaje_telegram = "🚨 *MATRIZ DE OPERACIONES (CALL / PUT)* 🚨\n\n"
    mensaje_telegram += "Fran, la bolsa abre a las 9:30 y el sistema ha analizado tus estrategias:\n\n"

    for _, r in ultimas_senales.iterrows():
        nombre_empresa = nombres_amigables.get(r['Ticker'], r['Ticker'])

        mensaje_telegram += f"Fran, hemos detectado la estrategia *{r['Estrategia']}* en *{nombre_empresa}*.\n" \
                            f"➡️ Dirección recomendada: *{r['Operacion']}*.\n" \
                            f"📅 Vela del: {r['Fecha']} (Precio: `${r['Precio']}`).\n\n" \
                            f"-----------------------------------\n\n"

    respuesta = enviar_alerta_telegram(mensaje_telegram)
    print("📲 ¡Alerta de Call/Put inteligente enviada a Telegram con éxito!")
else:
    print("No se encontraron señales de estrategia activas en este momento.")
print("==========================================================================")

In [ ]:
import io
import warnings
import matplotlib.pyplot as plt
import pandas as pd
import requests
import yfinance as yf

# Silenciar advertencias
warnings.simplefilter(action='ignore', category=FutureWarning)

# --- CONFIGURACIÓN DE TELEGRAM ---
p1 = "8556809936:AAFaziJLF1BOgnyXSBfJgt6S"
p2 = "J2asida9daE"
TELEGRAM_TOKEN = p1 + p2
CHAT_ID = "8642681599"

# Nombres amigables para las empresas y ETFs
nombres_amigables = {
    "SPY": "S&P 500 (SPY)",
    "QQQ": "Invesco QQQ (Tecnología)",
    "AAPL": "Apple",
    "AMZN": "Amazon",
    "MSFT": "Microsoft",
    "F": "Ford",
}


def enviar_foto_telegram(foto_bytes, caption):
  url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendPhoto"
  files = {"photo": ("chart.png", foto_bytes, "image/png")}
  data = {"chat_id": CHAT_ID, "caption": caption, "parse_mode": "Markdown"}
  try:
    response = requests.post(url, data=data, files=files)
    return response.json()
  except Exception as e:
    print(f"Error al enviar imagen a Telegram: {e}")


print("==========================================================================")
print("     MONITOR CON GRÁFICOS 16:9 Y FILTRO DE APERTURA (10:00 AM)             ")
print("==========================================================================")

senales_encontradas = 0

for ticker, nombre_empresa in nombres_amigables.items():
  try:
    # Descargamos datos de 30 minutos para ver la vela de 9:30 y la de 10:00
    df = yf.download(
        ticker, period="5d", interval="30m", progress=False, auto_adjust=False
    )
    if df.empty:
      continue
    if isinstance(df.columns, pd.MultiIndex):
      df.columns = df.columns.get_level_values(0)

    df.index = df.index.tz_convert("America/New_York")

    # Medias Móviles para la tendencia
    df["SMA_20"] = df["Close"].rolling(window=20).mean()
    df["SMA_40"] = df["Close"].rolling(window=40).mean()

    # Tomamos la última vela cerrada
    ultima_vela = df.iloc[-1]
    hora_vela = ultima_vela.name.strftime("%H:%M")
    fecha_vela = ultima_vela.name.strftime("%Y-%m-%d")

    # REGLA 1: Ignorar la vela de apertura de las 09:30
    if hora_vela == "09:30":
      continue

    # REGLA 2: A partir de las 10:00, comprobar si es alcista y cumple tendencia
    tendencia_alcista = (
        ultima_vela["Close"] > ultima_vela["SMA_20"]
    ) and (ultima_vela["SMA_20"] > ultima_vela["SMA_40"])
    es_verde = ultima_vela["Close"] > ultima_vela["Open"]

    if hora_vela >= "10:00" and es_verde and tendencia_alcista:
      precio_actual = round(ultima_vela["Close"], 2)

      # --- GENERAR GRÁFICO EN 16:9 ---
      plt.figure(
          figsize=(12, 6.75)
      )  # Proporción 16:9 exacta para tus preferencias
      ultimos_dias = df.tail(30)
      plt.plot(
          ultimos_dias.index,
          ultimos_dias["Close"],
          label="Precio",
          color="#1f77b4",
          linewidth=2,
      )
      plt.plot(
          ultimos_dias.index,
          ultimos_dias["SMA_20"],
          label="Media 20",
          color="#ff7f0e",
          linestyle="--",
      )

      # Marcar el punto de entrada exacto
      plt.scatter(
          [ultima_vela.name],
          [ultima_vela["Close"]],
          color="green",
          s=150,
          zorder=5,
          label="Entrada CALL",
      )

      plt.title(
          f"Estrategia CALL - {nombre_empresa} ({fecha_vela} {hora_vela})",
          fontsize=14,
          fontweight="bold",
      )
      plt.xlabel("Hora / Fecha", fontsize=10)
      plt.ylabel("Precio ($)", fontsize=10)
      plt.legend(loc="upper left")
      plt.grid(True, linestyle=":", alpha=0.6)
      plt.tight_layout()

      # Guardar gráfico en memoria RAM sin necesidad de guardarlo en disco duro
      buf = io.BytesIO()
      plt.savefig(buf, format="png", dpi=150)
      buf.seek(0)
      plt.close()

      # Mensaje para Telegram
      mensaje = (
          f"🚨 *¡SEÑAL DE CALL CONFIRMADA!* 🚨\n\n"
          f"Fran, ya pasó la apertura de las 9:30. La vela de las *{hora_vela}* en *{nombre_empresa}* ha cerrado verde y estamos en tendencia alcista.\n\n"
          f"➡️ *Acción:* Puedes comprar un *CALL* a partir de ${precio_actual}.\n"
          f"📅 *Fecha:* {fecha_vela}\n\n"
          f"Te adjunto el gráfico exacto del momento de entrada."
      )

      enviar_foto_telegram(buf.read(), mensaje)
      senales_encontradas += 1
      print(
          f"✅ Alerta con gráfico 16:9 enviada para {nombre_empresa} a las"
          f" {hora_vela}"
      )

  except Exception as e:
    continue

print(f"\n📊 Total de gráficos con señales enviados: {senales_encontradas}")
print("==========================================================================")


In [ ]:
!pip install mplfinance -q


In [ ]:
import io
import warnings
import matplotlib.pyplot as plt
import mplfinance as mpf
import pandas as pd
import requests
import yfinance as yf

warnings.filterwarnings('ignore')

TELEGRAM_TOKEN = "8556809936:AAFaziJLF1BOgnyXSBfJgt6SJ2asida9daE"
CHAT_ID = "8642681599"

def enviar_foto_telegram(foto_bytes, caption):
    url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendPhoto"
    files = {"photo": ("chart.png", foto_bytes, "image/png")}
    data = {"chat_id": CHAT_ID, "caption": caption, "parse_mode": "Markdown"}
    try:
        response = requests.post(url, data=data, files=files)
        return response.json()
    except Exception as e:
        print(f"Error: {e}")

nombres_amigables = {"SPY": "S&P 500", "AAPL": "Apple", "F": "Ford"}

for ticker, nombre in nombres_amigables.items():
    try:
        df = yf.download(ticker, period="5d", interval="30m", progress=False, auto_adjust=False)
        if df.empty: continue
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        df.index = df.index.tz_convert('America/New_York')
        ultima_vela = df.iloc[-1]

        if ultima_vela.name.strftime('%H:%M') == "09:30":
            continue

        fig, axes = mpf.plot(df.tail(20), type='candle', style='yahoo', returnfig=True, figsize=(12, 6.75))

        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=150)
        buf.seek(0)
        plt.close(fig)

        mensaje = f"🚨 *¡SEÑAL CONFIRMADA!* 🚨\nFran, en *{nombre}* el mercado dio entrada en la última vela. Te adjunto el gráfico de velas en tiempo real."

        enviar_foto_telegram(buf.read(), mensaje)
        print(f"✅ Alerta con velas enviada para {nombre}")
    except Exception as e:
        continue

In [ ]:
import io
import warnings
import matplotlib.pyplot as plt
import mplfinance as mpf
import pandas as pd
import requests
import yfinance as yf

# Silenciar advertencias
warnings.filterwarnings('ignore')

# --- CONFIGURACIÓN DE TELEGRAM ---
p1 = "8556809936:AAFaziJLF1BOgnyXSBfJgt6S"
p2 = "J2asida9daE"
TELEGRAM_TOKEN = p1 + p2
CHAT_ID = "8642681599"

# Lista completa de tus activos basada en tus capturas
activos_frances = {
    "SPY": "S&P 500 ETF (SPY)",
    "GLD": "SPDR Gold Shares (GLD)",
    "QQQ": "Invesco QQQ (Tecnología)",
    "META": "Meta Platforms (Facebook)",
    "NVDA": "NVIDIA Corporation",
    "AAPL": "Apple",
    "AMZN": "Amazon",
    "NFLX": "Netflix",
    "TSLA": "Tesla",
    "BAC": "Bank of America",
    "CMG": "Chipotle Mexican Grill",
    "SLV": "iShares Silver Trust (Plata)",
    "USO": "United States Oil Fund",
    "FB": "ProShares S&P 500",
}


def enviar_foto_telegram(foto_bytes, caption):
  url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendPhoto"
  files = {"photo": ("chart.png", foto_bytes, "image/png")}
  data = {"chat_id": CHAT_ID, "caption": caption, "parse_mode": "Markdown"}
  try:
    response = requests.post(url, data=data, files=files)
    return response.json()
  except Exception as e:
    print(f"Error al enviar imagen a Telegram: {e}")


print("==========================================================================")
print("     MONITOR TOTAL DE ACTIVOS - FILTRO 10:00 AM Y VELAS REALES            ")
print("==========================================================================")

reporte_general = (
    "📊 *REPORTE GENERAL DE TUS ACTIVOS* 📊\n\n"
    "Fran, la bolsa abrió y ya pasó el filtro de las 9:30. Aquí tienes el análisis de toda tu lista:\n\n"
)

activos_analizados = 0

for ticker, nombre in activos_frances.items():
  try:
    # Descargamos datos de 30 minutos
    df = yf.download(
        ticker, period="5d", interval="30m", progress=False, auto_adjust=False
    )
    if df.empty:
      continue
    if isinstance(df.columns, pd.MultiIndex):
      df.columns = df.columns.get_level_values(0)

    df.index = df.index.tz_convert("America/New_York")
    ultima_vela = df.iloc[-1]
    hora_vela = ultima_vela.name.strftime("%H:%M")

    # REGLA: Si es la vela de las 09:30, la ignoramos
    if hora_vela == "09:30":
      reporte_general += (
          f"📌 *{nombre}*: Esperando cierre de vela (última a las 09:30, sin"
          " operar).\n\n"
      )
      continue

    # Determinamos tendencia y dirección (Call / Put)
    open_p = ultima_vela["Open"]
    close_p = ultima_vela["Close"]
    precio_actual = round(close_p, 2)

    if close_p > open_p:
      direccion = "CALL 🟢 (Alcista / Vela Verde)"
    else:
      direccion = "PUT 🔴 (Bajista / Vela Roja)"

    reporte_general += (
        f"📌 *{nombre}* ({hora_vela}):\n➡️ Dirección: *{direccion}* a"
        f" `${precio_actual}`\n\n"
    )

    # Generamos el gráfico de velas para este activo
    fig, axes = mpf.plot(
        df.tail(15),
        type="candle",
        style="yahoo",
        returnfig=True,
        figsize=(12, 6.75),
    )

    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=150)
    buf.seek(0)
    plt.close(fig)

    # Enviamos la gráfica individual con su info a Telegram
    caption_activo = (
        f"📈 *{nombre}* - Vela de las {hora_vela}\nSugerencia: Operar"
        f" *{direccion}* a ${precio_actual}"
    )
    enviar_foto_telegram(buf.read(), caption_activo)
    activos_analizados += 1

  except Exception as e:
    continue

# Enviamos el reporte resumen completo al final
url_msg = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
requests.post(
    url_msg,
    json={"chat_id": CHAT_ID, "text": reporte_general, "parse_mode": "Markdown"},
)

print(
    f"\n✅ ¡Reporte completo enviado! Total de gráficos analizados:"
    f" {activos_analizados}"
)
print("==========================================================================")

In [ ]:
import io
import warnings
import matplotlib.pyplot as plt
import mplfinance as mpf
import pandas as pd
import requests
import yfinance as yf

# Silenciar advertencias
warnings.filterwarnings('ignore')

# --- CONFIGURACIÓN DE TELEGRAM ---
p1 = "8556809936:AAFaziJLF1BOgnyXSBfJgt6S"
p2 = "J2asida9daE"
TELEGRAM_TOKEN = p1 + p2
CHAT_ID = "8642681599"

# Lista completa de tus activos
activos_frances = {
    "SPY": "S&P 500 ETF (SPY)",
    "GLD": "SPDR Gold Shares (GLD)",
    "QQQ": "Invesco QQQ (Tecnología)",
    "META": "Meta Platforms (Facebook)",
    "NVDA": "NVIDIA Corporation",
    "AAPL": "Apple",
    "AMZN": "Amazon",
    "NFLX": "Netflix",
    "TSLA": "Tesla",
    "BAC": "Bank of America",
    "CMG": "Chipotle Mexican Grill",
    "SLV": "iShares Silver Trust (Plata)",
    "USO": "United States Oil Fund",
    "FB": "ProShares S&P 500",
}


def enviar_foto_telegram(foto_bytes, caption):
  url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendPhoto"
  files = {"photo": ("chart.png", foto_bytes, "image/png")}
  data = {"chat_id": CHAT_ID, "caption": caption, "parse_mode": "Markdown"}
  try:
    response = requests.post(url, data=data, files=files)
    return response.json()
  except Exception as e:
    print(f"Error al enviar imagen a Telegram: {e}")


print("==========================================================================")
print("     MONITOR TOTAL DE ACTIVOS - HORARIO ESTRICTO DE NUEVA YORK             ")
print("==========================================================================")

reporte_general = (
    "📊 *REPORTE GENERAL (HORARIO DE NUEVA YORK)* 📊\n\n"
    "Fran, aquí tienes el análisis de tus activos basado en la hora de apertura americana (NY):\n\n"
)

activos_analizados = 0

for ticker, nombre in activos_frances.items():
  try:
    # Descargamos datos de 30 minutos
    df = yf.download(
        ticker, period="5d", interval="30m", progress=False, auto_adjust=False
    )
    if df.empty:
      continue
    if isinstance(df.columns, pd.MultiIndex):
      df.columns = df.columns.get_level_values(0)

    # Forzamos la conversión estricta al horario de Nueva York
    df.index = df.index.tz_convert("America/New_York")
    ultima_vela = df.iloc[-1]
    hora_vela = ultima_vela.name.strftime("%H:%M")

    # REGLA ESTRICTA: Si es la vela de apertura de las 09:30 (Hora NY), se ignora
    if hora_vela == "09:30":
      reporte_general += (
          f"📌 *{nombre}*: Esperando cierre (última vela a las 09:30 NY, sin"
          " operar).\n\n"
      )
      continue

    open_p = ultima_vela["Open"]
    close_p = ultima_vela["Close"]
    precio_actual = round(close_p, 2)

    if close_p > open_p:
      direccion = "CALL 🟢 (Alcista / Vela Verde)"
    else:
      direccion = "PUT 🔴 (Bajista / Vela Roja)"

    reporte_general += (
        f"📌 *{nombre}* ({hora_vela} NY):\n➡️ Dirección: *{direccion}* a"
        f" `${precio_actual}`\n\n"
    )

    # Gráfico de velas
    fig, axes = mpf.plot(
        df.tail(15),
        type="candle",
        style="yahoo",
        returnfig=True,
        figsize=(12, 6.75),
    )

    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=150)
    buf.seek(0)
    plt.close(fig)

    caption_activo = (
        f"📈 *{nombre}* - Vela de las {hora_vela} (Hora NY)\nSugerencia:"
        f" Operar *{direccion}* a ${precio_actual}"
    )
    enviar_foto_telegram(buf.read(), caption_activo)
    activos_analizados += 1

  except Exception as e:
    continue

url_msg = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
requests.post(
    url_msg,
    json={"chat_id": CHAT_ID, "text": reporte_general, "parse_mode": "Markdown"},
)

print(
    f"\n✅ ¡Reporte en horario de Nueva York enviado! Gráficos:"
    f" {activos_analizados}"
)
print("==========================================================================")

In [ ]:
import warnings
import pandas as pd
import requests
import yfinance as yf

# Silenciar advertencias
warnings.filterwarnings('ignore')

# --- CONFIGURACIÓN DE TELEGRAM ---
p1 = "8556809936:AAFaziJLF1BOgnyXSBfJgt6S"
p2 = "J2asida9daE"
TELEGRAM_TOKEN = p1 + p2
CHAT_ID = "8642681599"

# Lista completa de tus activos
activos_frances = {
    "SPY": "S&P 500 ETF (SPY)",
    "GLD": "SPDR Gold Shares (GLD)",
    "QQQ": "Invesco QQQ (Tecnología)",
    "META": "Meta Platforms (Facebook)",
    "NVDA": "NVIDIA Corporation",
    "AAPL": "Apple",
    "AMZN": "Amazon",
    "NFLX": "Netflix",
    "TSLA": "Tesla",
    "BAC": "Bank of America",
    "CMG": "Chipotle Mexican Grill",
    "SLV": "iShares Silver Trust (Plata)",
    "USO": "United States Oil Fund",
    "FB": "ProShares S&P 500",
}


def enviar_mensaje_telegram(mensaje):
  url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
  payload = {"chat_id": CHAT_ID, "text": mensaje, "parse_mode": "Markdown"}
  try:
    response = requests.post(url, json=payload)
    return response.json()
  except Exception as e:
    print(f"Error al enviar mensaje a Telegram: {e}")


print("==========================================================================")
print("     REPORTE GENERAL ÚNICO - ANÁLISIS DE LA SEGUNDA VELA (10:00 NY)        ")
print("==========================================================================")

reporte_general = (
    "📊 *REPORTE GENERAL DE TUS ACTIVOS* 📊\n\n"
    "Fran, aquí tienes el resumen de cómo ha cerrado la *segunda vela del día (10:00 - 10:30 NY)* para cada activo:\n\n"
)

for ticker, nombre in activos_frances.items():
  try:
    # Descargamos datos recientes de 30 minutos
    df = yf.download(
        ticker, period="5d", interval="30m", progress=False, auto_adjust=False
    )
    if df.empty:
      continue
    if isinstance(df.columns, pd.MultiIndex):
      df.columns = df.columns.get_level_values(0)

    # Convertimos al horario de Nueva York
    df.index = df.index.tz_convert("America/New_York")

    # Buscamos la vela correspondiente a las 10:00 (la segunda vela, ignorando las 09:30)
    # Filtramos las velas del día actual o la última sesión disponible
    velas_hoy = df[df.index.date == df.index[-1].date()]

    # Verificamos si existe la vela de las 10:00
    vela_1000 = None
    for idx, row in velas_hoy.iterrows():
      if idx.strftime("%H:%M") == "10:00":
        vela_1000 = row
        break

    if vela_1000 is not None:
      open_p = vela_1000["Open"]
      close_p = vela_1000["Close"]
      precio = round(close_p, 2)

      if close_p > open_p:
        estado = "🟢 Positiva / Verde (¡Ideal para CALL si hay tendencia!)"
      else:
        estado = "🔴 Negativa / Roja"

      reporte_general += (
          f"📌 *{nombre}*:\n"
          f"   • 2ª Vela (10:00 NY): *{estado}*\n"
          f"   • Precio de cierre: `${precio}`\n\n"
      )
    else:
      reporte_general += (
          f"📌 *{nombre}*:\n   • Sin datos suficientes para la vela de las"
          " 10:00 NY.\n\n"
      )

  except Exception as e:
    continue

# Enviamos únicamente el reporte general consolidado a Telegram
enviar_mensaje_telegram(reporte_general)

print(
    "\n✅ ¡Reporte general único enviado con éxito a Telegram (sin spam"
    " individual)!"
)
print("==========================================================================")


In [ ]:
import warnings
import numpy as np
import pandas as pd
import requests
import yfinance as yf

# Silenciar advertencias
warnings.filterwarnings('ignore')

# --- CONFIGURACIÓN DE TELEGRAM ---
p1 = "8556809936:AAFaziJLF1BOgnyXSBfJgt6S"
p2 = "J2asida9daE"
TELEGRAM_TOKEN = p1 + p2
CHAT_ID = "8642681599"

# Lista completa de tus activos
activos_frances = {
    "SPY": "S&P 500 ETF (SPY)",
    "GLD": "SPDR Gold Shares (GLD)",
    "QQQ": "Invesco QQQ (Tecnología)",
    "META": "Meta Platforms (Facebook)",
    "NVDA": "NVIDIA Corporation",
    "AAPL": "Apple",
    "AMZN": "Amazon",
    "NFLX": "Netflix",
    "TSLA": "Tesla",
    "BAC": "Bank of America",
    "CMG": "Chipotle Mexican Grill",
    "SLV": "iShares Silver Trust (Plata)",
    "USO": "United States Oil Fund",
    "FB": "ProShares S&P 500",
}


def enviar_mensaje_telegram(mensaje):
  url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
  payload = {"chat_id": CHAT_ID, "text": mensaje, "parse_mode": "Markdown"}
  try:
    response = requests.post(url, json=payload)
    return response.json()
  except Exception as e:
    print(f"Error al enviar mensaje a Telegram: {e}")


print("==========================================================================")
print("     REPORTE INTELIGENTE: CANALES Y RECOMENDACIONES TÁCTICAS (NY)         ")
print("==========================================================================")

reporte_general = (
    "📊 *REPORTE TÁCTICO DE CANALES Y VELAS (NY)* 📊\n\n"
    "Fran, aquí tienes el análisis automatizado de canales (alcista/bajista) y las recomendaciones para operar:\n\n"
)

for ticker, nombre in activos_frances.items():
  try:
    # Descargamos datos históricos recientes de 30m para calcular canales
    df = yf.download(
        ticker, period="10d", interval="30m", progress=False, auto_adjust=False
    )
    if df.empty or len(df) < 20:
      continue
    if isinstance(df.columns, pd.MultiIndex):
      df.columns = df.columns.get_level_values(0)

    # Convertimos al horario de Nueva York
    df.index = df.index.tz_convert("America/New_York")

    # --- DETECCIÓN DE CANAL (Tendencia basada en los últimos periodos) ---
    # Usamos una regresión lineal simple o la pendiente de los cierres recientes (últimas 10 velas)
    cierres_recientes = df["Close"].tail(10).values
    x = np.arange(len(cierres_recientes))
    slope, _ = np.polyfit(x, cierres_recientes, 1)

    # Estimamos cuántas velas lleva en esta dirección (aproximando días/velas)
    fuerza_tendencia = "Canal Alcista 📈" if slope > 0 else "Canal Bajista 📉"
    duracion_velas = len(cierres_recientes)

    # Identificamos la segunda vela del día actual (10:00 NY)
    velas_hoy = df[df.index.date == df.index[-1].date()]
    vela_1000 = None
    for idx, row in velas_hoy.iterrows():
      if idx.strftime("%H:%M") == "10:00":
        vela_1000 = row
        break

    if vela_1000 is not None:
      open_p = vela_1000["Open"]
      close_p = vela_1000["Close"]
      precio = round(close_p, 2)
      es_verde = close_p > open_p

      # Lógica de recomendación personalizada estilo trading
      if slope < 0:
        # Canal Bajista
        if not es_verde:
          consejo = (
              "🔴 *Canal Bajista detectado*. Vela roja confirmada.\n   👉"
              " *Recomendación*: No recomiendo comprar CALL. *Ideal comprar PUT*."
          )
        else:
          consejo = (
              "⚠️ *Canal Bajista*, pero esta vela amagó verde.\n   👉"
              " *Recomendación*: Precaución con los CALLs, esperar confirmación"
              " bajista para PUT."
          )
      else:
        # Canal Alcista
        if es_verde:
          consejo = (
              "🟢 *Canal Alcista detectado*. Vela verde confirmada.\n   👉"
              " *Recomendación*: No recomiendo comprar PUT. *Ideal comprar CALL*."
          )
        else:
          consejo = (
              "⚠️ *Canal Alcista*, pero esta vela cerró roja.\n   👉"
              " *Recomendación*: Cuidado, posible retroceso en zona alta."
          )

      reporte_general += (
          f"📌 *{nombre}*:\n"
          f"   • Tendencia: *{fuerza_tendencia}*\n"
          f"   • 2ª Vela (10:00 NY): "
          f"{'🟢 Verde (Positiva)' if es_verde else '🔴 Roja (Negativa'})\n"
          f"   • Precio: `${precio}`\n"
          f"   {consejo}\n\n"
      )
    else:
      reporte_general += (
          f"📌 *{nombre}*:\n   • Tendencia: {fuerza_tendencia}\n   • Sin datos"
          " suficientes para la vela de las 10:00 NY.\n\n"
      )

  except Exception as e:
    continue

# Enviamos el reporte general inteligente a Telegram
enviar_mensaje_telegram(reporte_general)

print(
    "\n✅ ¡Reporte táctico de canales y recomendaciones enviado con éxito a"
    " Telegram!"
)
print("==========================================================================")

In [ ]:
import warnings
import numpy as np
import pandas as pd
import requests
import yfinance as yf

# Silenciar advertencias
warnings.filterwarnings('ignore')

# --- CONFIGURACIÓN DE TELEGRAM ---
p1 = "8556809936:AAFaziJLF1BOgnyXSBfJgt6S"
p2 = "J2asida9daE"
TELEGRAM_TOKEN = p1 + p2
CHAT_ID = "8642681599"

# Lista completa de tus activos
activos_frances = {
    "SPY": "S&P 500 ETF (SPY)",
    "GLD": "SPDR Gold Shares (GLD)",
    "QQQ": "Invesco QQQ (Tecnología)",
    "META": "Meta Platforms (Facebook)",
    "NVDA": "NVIDIA Corporation",
    "AAPL": "Apple",
    "AMZN": "Amazon",
    "NFLX": "Netflix",
    "TSLA": "Tesla",
    "BAC": "Bank of America",
    "CMG": "Chipotle Mexican Grill",
    "SLV": "iShares Silver Trust (Plata)",
    "USO": "United States Oil Fund",
    "FB": "ProShares S&P 500",
}


def enviar_mensaje_telegram(mensaje):
  url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
  payload = {"chat_id": CHAT_ID, "text": mensaje, "parse_mode": "Markdown"}
  try:
    response = requests.post(url, json=payload)
    return response.json()
  except Exception as e:
    print(f"Error al enviar mensaje a Telegram: {e}")


print("==========================================================================")
print("     REPORTE INTELIGENTE DE CIERRE Y PERSPECTIVAS FUTURAS (NY)            ")
print("==========================================================================")

reporte_general = (
    f"🔔 *REPORTE DE CIERRE DE MERCADO Y PERSPECTIVAS* 🔔\n\n"
    f"Hola *Franz*. Aquí tienes el análisis global del comportamiento de tus activos al cierre de la sesión y las proyecciones para futuras inversiones:\n\n"
    f"━━━━━━━━━━━━━━━━━━━━━━\n"
    f"📈 *ANÁLISIS DE LA SESIÓN Y TENDENCIAS*:\n\n"
)

resumen_proyecciones = (
    f"━━━━━━━━━━━━━━━━━━━━━━\n"
    f"🎯 *PUNTO DE VISTA PARA FUTURAS INVERSIONES (CALL / PUT)*:\n\n"
)

for ticker, nombre in activos_frances.items():
  try:
    # Descargamos datos históricos recientes de 30m para evaluar canal y cierre del día
    df = yf.download(
        ticker, period="10d", interval="30m", progress=False, auto_adjust=False
    )
    if df.empty or len(df) < 20:
      continue
    if isinstance(df.columns, pd.MultiIndex):
      df.columns = df.columns.get_level_values(0)

    # Convertimos al horario de Nueva York
    df.index = df.index.tz_convert("America/New_York")

    # Tendencia general (Canal)
    cierres_recientes = df["Close"].tail(10).values
    x = np.arange(len(cierres_recientes))
    slope, _ = np.polyfit(x, cierres_recientes, 1)

    tendencia = "Canal Alcista 📈" if slope > 0 else "Canal Bajista 📉"

    # Evaluamos la última vela del día (cierre)
    ultima_vela = df.iloc[-1]
    open_cierre = ultima_vela["Open"]
    close_cierre = ultima_vela["Close"]
    precio_cierre = round(close_cierre, 2)
    cierre_verde = close_cierre > open_cierre

    # Añadimos al reporte de sesión
    reporte_general += (
        f"📌 *{nombre}*:\n"
        f"   • Cierre: `${precio_cierre}` ({'Verde 🟢' if cierre_verde else 'Roja 🔴'})\n"
        f"   • Estructura: *{tendencia}*\n\n"
    )

    # Generamos la perspectiva para futuras inversiones
    if slope < 0:
      # Si está en bajista
      perspectiva = (
          f"🔸 *{nombre}* (Bajista): El activo mantiene presión vendedora. "
          f"Evitar buscar CALLs precipitados. *Vigilar la primera vela roja en próxima sesión para posibles entradas en PUT*.\n\n"
      )
    else:
      # Si está en alcista
      perspectiva = (
          f"🔸 *{nombre}* (Alcista): Estructura favorable al alza. "
          f"Evitar compras en PUT contra tendencia. *Atentos a correcciones para buscar oportunidades en CALL*.\n\n"
      )

    resumen_proyecciones += perspectiva

  except Exception as e:
    continue

# Unimos ambas partes en un solo mensaje estructurado
mensaje_final = reporte_general + resumen_proyecciones

# Enviamos el reporte general consolidado de cierre a Telegram
enviar_mensaje_telegram(mensaje_final)

print(
    "\n✅ ¡Reporte global de cierre y proyecciones enviado correctamente a"
    " Telegram, Franz!"
)
print("==========================================================================")

In [ ]:
import warnings
import numpy as np
import pandas as pd
import requests
import yfinance as yf

# Silenciar advertencias
warnings.filterwarnings('ignore')

# --- CONFIGURACIÓN DE TELEGRAM ---
p1 = "8556809936:AAFaziJLF1BOgnyXSBfJgt6S"
p2 = "J2asida9daE"
TELEGRAM_TOKEN = p1 + p2
CHAT_ID = "8642681599"

# Lista completa de tus activos
activos_frances = {
    "SPY": "S&P 500 ETF (SPY)",
    "GLD": "SPDR Gold Shares (GLD)",
    "QQQ": "Invesco QQQ (Tecnología)",
    "META": "Meta Platforms (Facebook)",
    "NVDA": "NVIDIA Corporation",
    "AAPL": "Apple",
    "AMZN": "Amazon",
    "NFLX": "Netflix",
    "TSLA": "Tesla",
    "BAC": "Bank of America",
    "CMG": "Chipotle Mexican Grill",
    "SLV": "iShares Silver Trust (Plata)",
    "USO": "United States Oil Fund",
    "FB": "ProShares S&P 500",
}


def enviar_alerta_telegram(mensaje):
  url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
  payload = {"chat_id": CHAT_ID, "text": mensaje, "parse_mode": "Markdown"}
  try:
    response = requests.post(url, json=payload)
    return response.json()
  except Exception as e:
    print(f"Error al enviar Telegram: {e}")


print("==========================================================================")
print("     MONITOR TÁCTICO EN TIEMPO REAL - ALERTAS DE ENTRADA (NY)             ")
print("==========================================================================")

alertas_enviadas = 0

for ticker, nombre in activos_frances.items():
  try:
    # Descargamos datos recientes de 30m
    df = yf.download(
        ticker, period="5d", interval="30m", progress=False, auto_adjust=False
    )
    if df.empty or len(df) < 20:
      continue
    if isinstance(df.columns, pd.MultiIndex):
      df.columns = df.columns.get_level_values(0)

    # Convertimos estrictamente al horario de Nueva York
    df.index = df.index.tz_convert("America/New_York")

    # Analizamos la última vela cerrada
    ultima_vela = df.iloc[-1]
    hora_vela = ultima_vela.name.strftime("%H:%M")

    # REGLA DE ORO DE FRANZ: Ignoramos la vela de apertura de las 09:30 a 10:00
    if hora_vela == "09:30":
      continue

    # Calculamos el canal (tendencia de las últimas 10 velas)
    cierres_recientes = df["Close"].tail(10).values
    x = np.arange(len(cierres_recientes))
    slope, _ = np.polyfit(x, cierres_recientes, 1)
    en_canal_bajista = slope < 0

    open_p = ultima_vela["Open"]
    close_p = ultima_vela["Close"]
    precio_actual = round(close_p, 2)
    es_vela_verde = close_p > open_p

    # DISPARADOR DE ESTRATEGIA EN TIEMPO REAL
    # Si estamos a las 10:00 o más tarde, evaluamos tus reglas exactas:
    if hora_vela >= "10:00":
      if en_canal_bajista and not es_vela_verde:
        # Canal bajista + vela roja que cierra = ¡Oportunidad de PUT!
        alerta = (
            f"🚨 *¡ALERTA TÁCTICA PARA FRANZ!* 🚨\n\n"
            f"⚡ *Activo*: *{nombre}*\n"
            f"⏰ *Hora NY*: {hora_vela} (Vela cerrada)\n"
            f"📉 *Estructura*: Canal bajista activo.\n\n"
            f"👉 *Franz, corre*: La vela acaba de cerrar *ROJA* a `${precio_actual}` dentro del canal bajista.\n"
            f"❌ *No compres CALL*.\n"
            f"🎯 *Acción sugerida*: Es el momento idóneo para buscar una entrada en **PUT**."
        )
        enviar_alerta_telegram(alerta)
        alertas_enviadas += 1

      elif not en_canal_bajista and es_vela_verde:
        # Canal alcista + vela verde que cierra = ¡Oportunidad de CALL!
        alerta = (
            f"🚨 *¡ALERTA TÁCTICA PARA FRANZ!* 🚨\n\n"
            f"⚡ *Activo*: *{nombre}*\n"
            f"⏰ *Hora NY*: {hora_vela} (Vela cerrada)\n"
            f"📈 *Estructura*: Canal alcista activo.\n\n"
            f"👉 *Franz, corre*: La vela acaba de cerrar *VERDE* a `${precio_actual}` en tendencia alcista.\n"
            f"❌ *No compres PUT*.\n"
            f"🎯 *Acción sugerida*: Es el momento idóneo para buscar una entrada en **CALL**."
        )
        enviar_alerta_telegram(alerta)
        alertas_enviadas += 1

  except Exception as e:
    continue

print(
    f"\n✅ Análisis completado, Franz. Alertas tácticas enviadas:"
    f" {alertas_enviadas}"
)
print("==========================================================================")

In [ ]:
import warnings
import numpy as np
import pandas as pd
import requests
import yfinance as yf

# Silenciar advertencias
warnings.filterwarnings('ignore')

# --- CONFIGURACIÓN DE TELEGRAM ---
p1 = "8556809936:AAFaziJLF1BOgnyXSBfJgt6S"
p2 = "J2asida9daE"
TELEGRAM_TOKEN = p1 + p2
CHAT_ID = "8642681599"

# Lista completa de tus activos
activos_frances = {
    "SPY": "S&P 500 ETF (SPY)",
    "GLD": "SPDR Gold Shares (GLD)",
    "QQQ": "Invesco QQQ (Tecnología)",
    "META": "Meta Platforms (Facebook)",
    "NVDA": "NVIDIA Corporation",
    "AAPL": "Apple",
    "AMZN": "Amazon",
    "NFLX": "Netflix",
    "TSLA": "Tesla",
    "BAC": "Bank of America",
    "CMG": "Chipotle Mexican Grill",
    "SLV": "iShares Silver Trust (Plata)",
    "USO": "United States Oil Fund",
    "FB": "ProShares S&P 500",
}


def enviar_alerta_telegram(mensaje):
  url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
  payload = {"chat_id": CHAT_ID, "text": mensaje, "parse_mode": "Markdown"}
  try:
    response = requests.post(url, json=payload)
    return response.json()
  except Exception as e:
    print(f"Error al enviar Telegram: {e}")


print("==========================================================================")
print("     MONITOR TÁCTICO DAY TRADING - ANÁLISIS AUTOMATIZADO (NY)             ")
print("==========================================================================")

reporte_general = (
    f"🤖 *REPORTE AUTOMATIZADO DE DAY TRADING* 🤖\n\n"
    f"Hola *Franz*. Aquí tienes el escaneo en tiempo real de tus activos con las señales de opciones (Call/Put) basadas en tus estrategias:\n\n"
)

senales_encontradas = 0

for ticker, nombre in activos_frances.items():
  try:
    # Descargamos datos recientes de 30m para day trading
    df = yf.download(
        ticker, period="5d", interval="30m", progress=False, auto_adjust=False
    )
    if df.empty or len(df) < 20:
      continue
    if isinstance(df.columns, pd.MultiIndex):
      df.columns = df.columns.get_level_values(0)

    # Convertimos estrictamente al horario de Nueva York
    df.index = df.index.tz_convert("America/New_York")

    # Analizamos la última vela cerrada
    ultima_vela = df.iloc[-1]
    hora_vela = ultima_vela.name.strftime("%H:%M")

    # Evitamos la vela de apertura inicial de las 09:30
    if hora_vela == "09:30":
      continue

    # Detección del canal o tendencia (últimas 10 velas)
    cierres_recientes = df["Close"].tail(10).values
    x = np.arange(len(cierres_recientes))
    slope, _ = np.polyfit(x, cierres_recientes, 1)
    en_canal_bajista = slope < 0

    open_p = ultima_vela["Open"]
    close_p = ultima_vela["Close"]
    precio_actual = round(close_p, 2)
    es_vela_verde = close_p > open_p

    # Lógica automatizada para Day Trading (Operaciones en corto / intradía)
    if en_canal_bajista:
      if not es_vela_verde:
        estrategia = (
            "📉 *Señal Bajista Confirmada*:\n"
            "   • Canal bajista activo y vela cerrada *Roja*.\n"
            "   👉 *Estrategia Day Trading*: Evitar CALL. *Oportunidad ideal para PUT* a"
            f" `${precio_actual}`."
        )
        senales_encontradas += 1
      else:
        estrategia = (
            "⚠️ *Atención Bajista*:\n"
            "   • Canal bajista pero la vela cerró verde (posible amague).\n"
            "   👉 *Estrategia Day Trading*: Esperar confirmación para PUT, no arriesgar CALL."
        )
    else:
      if es_vela_verde:
        estrategia = (
            "📈 *Señal Alcista Confirmada*:\n"
            "   • Canal alcista activo y vela cerrada *Verde*.\n"
            "   👉 *Estrategia Day Trading*: Evitar PUT. *Oportunidad ideal para CALL* a"
            f" `${precio_actual}`."
        )
        senales_encontradas += 1
      else:
        estrategia = (
            "⚠️ *Atención Alcista*:\n"
            "   • Canal alcista pero la vela cerró roja (posible retroceso).\n"
            "   👉 *Estrategia Day Trading*: Precaución con los CALLs, vigilar soporte."
        )

    reporte_general += (
        f"📌 *{nombre}* ({hora_vela} NY):\n"
        f"   • Precio actual: `${precio_actual}`\n"
        f"   {estrategia}\n\n"
    )

  except Exception as e:
    continue

# Enviamos el reporte automatizado con las recomendaciones integradas
enviar_alerta_telegram(reporte_general)

print(
    f"\n✅ Análisis completado, Franz. Señales de Day Trading procesadas y"
    f" enviadas: {senales_encontradas}"
)
print("==========================================================================")